# LLMEmbed++ — Exp 0–5 trên Kaggle

Notebook tự chứa mã nguồn, không cần clone repo. **Settings → GPU T4 ×2; Internet → On.**
Với LLaMA-2, chấp nhận điều kiện model trên Hugging Face và tạo Kaggle Secret `HF_TOKEN` có quyền đọc model.

Mặc định chạy thử **256 mẫu, 2 epochs, seed 42, NF4** trên SST-2. Đổi cấu hình bên dưới để chạy đầy đủ.
NF4 là biến thể tiết kiệm VRAM; dùng FP16 cho đối chiếu không lượng tử hóa. Không có kết quả dự đoán sẵn.


In [ ]:
from pathlib import Path
import json, os, sys, subprocess
CODE = Path('/kaggle/working/llmembed_code')
CODE.mkdir(parents=True, exist_ok=True)
FILES = json.loads('{"llmembed/__init__.py": "\\"\\"\\"Frozen-backbone LLMEmbed experiments for Kaggle.\\"\\"\\"\\n", "llmembed/data.py": "\\"\\"\\"Dataset provenance and deterministic train-only validation splits.\\"\\"\\"\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\nfrom urllib.request import urlopen\\nimport warnings\\n\\nimport numpy as np\\n\\nEXPECTED = {\'sst2\': (67349, 872, 2), \'mr\': (40000, 10000, 2),\\n            \'agnews\': (120000, 7600, 4), \'r8\': (5485, 2189, 8), \'r52\': (6532, 2568, 52)}\\nTEXTGCN_COMMIT = \'962223652e9bb164ac2d83cd09fc7b8845ce860b\'\\n\\n\\ndef digest(value):\\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()\\n\\n\\ndef write_json(path, value):\\n    path = Path(path)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding=\'utf-8\')\\n    temporary.replace(path)\\n\\n\\ndef normalize_records(data):\\n    if isinstance(data, dict):\\n        texts, labels = data.get(\'text\', data.get(\'sentence\')), data.get(\'label\')\\n        if not isinstance(texts, list) or not isinstance(labels, list) or len(texts) != len(labels):\\n            raise ValueError(\'Expected equally sized text and label arrays.\')\\n        data = [dict(text=t, label=y) for t, y in zip(texts, labels)]\\n    if not isinstance(data, list) or not data:\\n        raise ValueError(\'Dataset must be a nonempty list of {text, label} records.\')\\n    result = []\\n    for i, row in enumerate(data):\\n        if not isinstance(row, dict):\\n            raise ValueError(f\'Expected an object at row {i}.\')\\n        text, label = row.get(\'text\', row.get(\'sentence\')), row.get(\'label\')\\n        if not isinstance(text, str) or not text.strip():\\n            raise ValueError(f\'Empty/non-string text at row {i}.\')\\n        if not isinstance(label, (str, int)) or isinstance(label, bool) or str(label) in (\'\', \'-1\'):\\n            raise ValueError(f\'Missing/invalid label at row {i}: {label!r}\')\\n        result.append(dict(text=text, label=str(label)))\\n    return result\\n\\n\\ndef split_indices(labels, fraction, seed):\\n    \\"\\"\\"Keep singleton classes in training; every other class contributes validation.\\"\\"\\"\\n    if not 0 <= fraction < 1:\\n        raise ValueError(\'Validation fraction must be in [0, 1).\')\\n    rng = np.random.default_rng(seed)\\n    labels = np.asarray(labels)\\n    train, val = [], []\\n    for label in np.unique(labels):\\n        indices = np.flatnonzero(labels == label)\\n        rng.shuffle(indices)\\n        n = min(len(indices) - 1, max(1, round(len(indices) * fraction))) if fraction else 0\\n        val.extend(indices[:n].tolist())\\n        train.extend(indices[n:].tolist())\\n    return sorted(train), sorted(val)\\n\\n\\ndef subset(rows, limit, seed):\\n    if not limit or limit >= len(rows):\\n        return rows\\n    labels = np.array([r[\'label\'] for r in rows])\\n    rng = np.random.default_rng(seed)\\n    # Round-robin classes makes a small smoke run useful even on imbalanced R52.\\n    groups = [rng.permutation(np.flatnonzero(labels == y)).tolist() for y in np.unique(labels)]\\n    chosen = []\\n    while len(chosen) < limit:\\n        for group in groups:\\n            if group and len(chosen) < limit:\\n                chosen.append(group.pop())\\n    return [rows[i] for i in sorted(chosen)]\\n\\n\\ndef load_raw(dataset, data_dir, download_dir, mr_source):\\n    local = Path(data_dir) / dataset if data_dir else None\\n    if local and local.exists():\\n        def read(split):\\n            path = local / f\'{split}.json\'\\n            if path.exists():\\n                return normalize_records(json.loads(path.read_text(encoding=\'utf-8\')))\\n            path = local / f\'{split}.jsonl\'\\n            return normalize_records([json.loads(s) for s in path.read_text(encoding=\'utf-8\').splitlines() if s.strip()])\\n        return read(\'train\'), read(\'test\'), f\'local:{local.resolve()}\', \'local_split_unverified\'\\n    if dataset in (\'r8\', \'r52\'):\\n        base = f\'https://raw.githubusercontent.com/yao8839836/text_gcn/{TEXTGCN_COMMIT}/data/\'\\n        parts = []\\n        for filename in (f\'{dataset.upper()}.txt\', f\'corpus/{dataset.upper()}.txt\'):\\n            target = Path(download_dir) / filename\\n            if not target.exists():\\n                target.parent.mkdir(parents=True, exist_ok=True)\\n                with urlopen(base + filename, timeout=120) as response:\\n                    content = response.read()\\n                temp = target.with_suffix(\'.tmp\')\\n                temp.write_bytes(content)\\n                temp.replace(target)\\n            parts.append(target.read_text(encoding=\'utf-8\').splitlines())\\n        metadata, texts = parts\\n        if len(metadata) != len(texts):\\n            raise ValueError(\'TextGCN text/metadata row counts differ.\')\\n        splits = {\'train\': [], \'test\': []}\\n        for meta, text in zip(metadata, texts):\\n            _, split, label = meta.split(\'\\\\t\')\\n            splits[split].append(dict(text=text, label=label))\\n        return (normalize_records(splits[\'train\']), normalize_records(splits[\'test\']),\\n                base, \'textgcn_split_preprocessing_not_verified_against_llmembed\')\\n    from datasets import load_dataset\\n    if dataset == \'mr\':\\n        if mr_source != \'imdb_resplit\':\\n            raise ValueError(\'Exact MR split is unavailable. Supply data/mr/train.json and test.json, \'\\n                             \'or explicitly select --mr-source imdb_resplit (different benchmark split).\')\\n        warnings.warn(\'MR proxy: IMDb re-split 40k/10k, seed 42; NOT the verified LLMEmbed MR split.\')\\n        source = load_dataset(\'stanfordnlp/imdb\')\\n        rows = normalize_records([dict(text=r[\'text\'], label=r[\'label\'])\\n                                  for split in (\'train\', \'test\') for r in source[split]])\\n        train, test = split_indices([r[\'label\'] for r in rows], 0.2, 42)\\n        return [rows[i] for i in train], [rows[i] for i in test], \'stanfordnlp/imdb:80-20:seed42\', \'imdb_resplit_proxy\'\\n    name, field, test_split = (\'stanfordnlp/sst2\', \'sentence\', \'validation\') if dataset == \'sst2\' else (\'fancyzhx/ag_news\', \'text\', \'test\')\\n    source = load_dataset(name)\\n    rows = [normalize_records([dict(text=r[field], label=r[\'label\']) for r in source[s]])\\n            for s in (\'train\', test_split)]\\n    return *rows, name, f\'huggingface_train_{test_split}\'\\n\\n\\ndef prepare_data(dataset, data_dir, output_dir, limit=0, seed=42, val_fraction=0.1, mr_source=\'local\'):\\n    if dataset not in EXPECTED or limit < 0:\\n        raise ValueError(\'Unknown dataset or negative sample limit.\')\\n    train, test, source, status = load_raw(dataset, data_dir, Path(output_dir) / \'downloads\', mr_source)\\n    names = sorted({r[\'label\'] for r in train})\\n    mapping = {name: i for i, name in enumerate(names)}\\n    raw_counts = (len(train), len(test), len(names))\\n    raw_hash = digest([train, test])\\n    for split, rows in [(\'train\', train), (\'test\', test)]:\\n        for i, row in enumerate(rows):\\n            if row[\'label\'] not in mapping:\\n                raise ValueError(f\'Test label absent from train: {row[\\"label\\"]}\')\\n            row.update(id=f\'{split}:{i}\', label=mapping[row[\'label\']])\\n    train, test = subset(train, limit, seed), subset(test, limit, seed)\\n    tr, val = split_indices([r[\'label\'] for r in train], val_fraction, seed)\\n    if val_fraction and not val:\\n        raise ValueError(\'Not enough training examples for validation; increase --limit.\')\\n    splits = dict(train=[train[i] for i in tr], val=[train[i] for i in val], test=test)\\n    train_texts = {digest(r[\'text\']) for r in train}\\n    overlap = sum(digest(r[\'text\']) in train_texts for r in test)\\n    metadata = dict(dataset=dataset, source=source, benchmark_status=status, raw_sha256=raw_hash,\\n                    label_names=names, original_counts=raw_counts, paper_counts=EXPECTED[dataset],\\n                    counts_match_paper=raw_counts == EXPECTED[dataset], split_seed=seed,\\n                    val_fraction=val_fraction, limit=limit, exact_text_overlap_train_test=overlap)\\n    if overlap:\\n        warnings.warn(f\'{dataset}: {overlap} test texts also occur in train; preserving the source split, recorded in manifest.\')\\n    identity = dict(metadata, source=\'local\' if source.startswith(\'local:\') else source)\\n    metadata[\'fingerprint\'] = digest([identity, splits])\\n    target = Path(output_dir) / \'data\' / f\'{dataset}-{metadata[\\"fingerprint\\"][:16]}\'\\n    write_json(target / \'manifest.json\', metadata)\\n    for split, rows in splits.items():\\n        write_json(target / f\'{split}.json\', rows)\\n    return dict(metadata, splits=splits)\\n", "llmembed/features.py": "\\"\\"\\"Masked extraction and append-only, resumable float16 feature caches.\\"\\"\\"\\nimport gc\\nimport json\\nimport math\\nimport os\\nfrom pathlib import Path\\nimport time\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn.functional as F\\n\\nfrom .data import digest, write_json\\n\\n\\ndef window_ranges(length, size, overlap, max_chunks=0):\\n    if length < 1 or size < 1 or not 0 <= overlap < size or max_chunks < 0:\\n        raise ValueError(\'Require nonempty text, size > overlap >= 0 and max_chunks >= 0.\')\\n    ranges, start = [], 0\\n    while start < length:\\n        end = min(start + size, length)\\n        ranges.append((start, end))\\n        if end == length:\\n            break\\n        start = end - overlap\\n    if max_chunks and len(ranges) > max_chunks:\\n        ranges = [ranges[i] for i in np.linspace(0, len(ranges) - 1, max_chunks, dtype=int)]\\n    return ranges\\n\\n\\ndef make_windows(tokenizer, text, spec):\\n    ids = tokenizer.encode(text, add_special_tokens=False, verbose=False)\\n    if not ids:\\n        raise ValueError(\'Text tokenized to an empty sequence.\')\\n    prefix, suffix = [], []\\n    if spec[\'pooling\'] == \'cot\':\\n        sentiment = spec[\'task\'] in (\'sst2\', \'mr\')\\n        prefix = tokenizer.encode(\'Text: \\"\' if sentiment else \'Article: \\"\', add_special_tokens=False)\\n        suffix = tokenizer.encode(\'\\". After analyzing the key sentiment clues step by step, the overall sentiment of this text in one word is:\' if sentiment else\\n                                  \'\\". After identifying the core domain clues and reasoning step by step, the topic category of this article in one word is:\', add_special_tokens=False)\\n    budget = spec[\'max_length\'] - len(prefix) - len(suffix) - tokenizer.num_special_tokens_to_add(pair=False)\\n    if budget < 1:\\n        raise ValueError(\'max_length cannot fit the prompt and one content token.\')\\n    ranges = (window_ranges(len(ids), min(spec[\'chunk_size\'], budget), spec[\'overlap\'], spec[\'max_chunks\'])\\n              if spec[\'chunked\'] else [(0, min(len(ids), budget))])\\n    entries = []\\n    for start, end in ranges:\\n        body = prefix + ids[start:end] + suffix\\n        full = tokenizer.build_inputs_with_special_tokens(body)\\n        # Supported single-sequence tokenizers add specials only at the boundaries.\\n        offset = next((i for i in range(len(full) - len(body) + 1) if full[i:i + len(body)] == body), None)\\n        if offset is None:\\n            raise ValueError(\'Tokenizer inserted special tokens inside the input sequence.\')\\n        mask = [False] * len(full)\\n        mask[offset + len(prefix):offset + len(prefix) + end - start] = [True] * (end - start)\\n        entries.append(dict(input_ids=full, content_mask=mask, anchor=offset + len(body) - 1,\\n                            start=start, end=end))\\n    retained, previous = 0, 0\\n    for start, end in ranges:\\n        retained += max(0, end - max(start, previous))\\n        previous = end\\n    return entries, len(ids), retained\\n\\n\\ndef token_pool(hidden, attention, content, anchors, mode, gamma=0.5, tau=0.1):\\n    if tau <= 0 or not 0 <= gamma <= 1:\\n        raise ValueError(\'Require tau > 0 and 0 <= gamma <= 1.\')\\n    hidden = hidden.float()\\n    attention = attention.to(hidden.device).bool()\\n    if mode == \'legacy_mean\':\\n        return hidden.mean(1), None\\n    if mode == \'mean\':\\n        return (hidden * attention.unsqueeze(-1)).sum(1) / attention.sum(1, keepdim=True).clamp_min(1), None\\n    if mode != \'cot\':\\n        raise ValueError(f\'Unknown pooling: {mode}\')\\n    content = content.to(hidden.device).bool() & attention\\n    if not content.any(1).all():\\n        raise ValueError(\'Attention pooling needs at least one original-text token.\')\\n    anchor = hidden[torch.arange(len(hidden), device=hidden.device), anchors.to(hidden.device)]\\n    scores = (F.normalize(hidden, dim=-1) * F.normalize(anchor, dim=-1).unsqueeze(1)).sum(-1) / tau\\n    weights = scores.masked_fill(~content, -torch.inf).softmax(1)\\n    return gamma * anchor + (1 - gamma) * (weights.unsqueeze(-1) * hidden).sum(1), weights\\n\\n\\nclass FeatureStore:\\n    def __init__(self, path, signature, shape, documents):\\n        self.path, self.shape = Path(path), tuple(shape)\\n        self.path.mkdir(parents=True, exist_ok=True)\\n        self.binary, self.manifest = self.path / \'values.bin\', self.path / \'state.json\'\\n        self.array = None\\n        if self.manifest.exists():\\n            self.state = json.loads(self.manifest.read_text(encoding=\'utf-8\'))\\n            if self.state[\'signature\'] != signature or self.state[\'shape\'] != list(shape) or self.state[\'documents\'] != documents:\\n                raise ValueError(\'Cache signature mismatch; select another output directory.\')\\n        else:\\n            self.state = dict(signature=signature, shape=list(shape), documents=documents, offsets=[0],\\n                              seconds=0.0, original_tokens=0, retained_tokens=0, explanations=[], complete=False)\\n        expected = self.state[\'offsets\'][-1] * math.prod(shape) * 2\\n        actual = self.binary.stat().st_size if self.binary.exists() else 0\\n        if actual < expected or (self.state[\'complete\'] and actual != expected):\\n            raise ValueError(f\'Truncated/corrupt feature cache: {self.path}\')\\n        if actual > expected and not self.state[\'complete\']:\\n            with self.binary.open(\'r+b\') as stream:\\n                stream.truncate(expected)\\n\\n    @property\\n    def done(self):\\n        return len(self.state[\'offsets\']) - 1\\n\\n    def append(self, documents, seconds, original, retained, explanations):\\n        if self.state[\'complete\'] or self.done + len(documents) > self.state[\'documents\']:\\n            raise ValueError(\'Cannot append beyond the planned document count.\')\\n        arrays = [np.asarray(a, dtype=np.float16) for a in documents]\\n        if any(a.ndim != 3 or a.shape[1:] != self.shape or len(a) == 0 or not np.isfinite(a).all() for a in arrays):\\n            raise ValueError(\'Invalid/non-finite extracted feature array.\')\\n        with self.binary.open(\'ab\') as stream:\\n            for array in arrays:\\n                array.tofile(stream)\\n                self.state[\'offsets\'].append(self.state[\'offsets\'][-1] + len(array))\\n            stream.flush()\\n            os.fsync(stream.fileno())\\n        self.state[\'seconds\'] += seconds\\n        self.state[\'original_tokens\'] += original\\n        self.state[\'retained_tokens\'] += retained\\n        self.state[\'explanations\'].extend(explanations)\\n        write_json(self.manifest, self.state)\\n\\n    def finish(self):\\n        if self.done != self.state[\'documents\']:\\n            raise ValueError(\'Cannot finish an incomplete feature cache.\')\\n        self.state[\'complete\'] = True\\n        write_json(self.manifest, self.state)\\n\\n    def __getitem__(self, index):\\n        if not self.state[\'complete\']:\\n            raise ValueError(\'Training requires a completed feature cache.\')\\n        if self.array is None:\\n            self.array = np.memmap(self.binary, dtype=np.float16, mode=\'r\',\\n                                   shape=(self.state[\'offsets\'][-1], *self.shape))\\n        return self.array[self.state[\'offsets\'][index]:self.state[\'offsets\'][index + 1]]\\n\\n    def close(self):\\n        self.array = None\\n\\n\\ndef encode_batch(model, tokenizer, entries, spec):\\n    length = spec[\'max_length\'] if spec[\'pooling\'] == \'legacy_mean\' else max(len(e[\'input_ids\']) for e in entries)\\n    ids = torch.full((len(entries), length), tokenizer.pad_token_id, dtype=torch.long)\\n    attention = torch.zeros_like(ids)\\n    content = torch.zeros_like(ids, dtype=torch.bool)\\n    for i, entry in enumerate(entries):\\n        n = len(entry[\'input_ids\'])\\n        ids[i, :n] = torch.tensor(entry[\'input_ids\'])\\n        attention[i, :n] = 1\\n        content[i, :n] = torch.tensor(entry[\'content_mask\'])\\n    device = model.get_input_embeddings().weight.device\\n    with torch.inference_mode():\\n        kwargs = dict(input_ids=ids.to(device), attention_mask=attention.to(device), return_dict=True)\\n        if spec[\'source\'] == \'llama\':\\n            kwargs.update(output_hidden_states=True, use_cache=False)\\n        outputs = model(**kwargs)\\n        if spec[\'source\'] != \'llama\':\\n            values = outputs.pooler_output if spec[\'pooling\'] == \'pooler\' else outputs.last_hidden_state[:, 0]\\n            return values[:, None].float().cpu().numpy(), None\\n        anchors = torch.tensor([e[\'anchor\'] for e in entries])\\n        features, alpha = [], None\\n        for depth in spec[\'layers\']:\\n            value, weights = token_pool(outputs.hidden_states[depth], attention, content, anchors,\\n                                        spec[\'pooling\'], spec[\'gamma\'], spec[\'tau\'])\\n            features.append(value.cpu())\\n            if alpha is None and weights is not None:\\n                alpha = weights.cpu().numpy()\\n        return torch.stack(features, 1).numpy(), alpha\\n\\n\\ndef extract_rows(model, tokenizer, rows, spec, store, batch_size=2, shard_docs=32, explain=4):\\n    while store.done < len(rows):\\n        if torch.cuda.is_available():\\n            torch.cuda.synchronize()\\n        started = time.perf_counter()\\n        start = store.done\\n        selected = rows[start:start + shard_docs]\\n        entries, counts, original, retained = [], [], 0, 0\\n        for row in selected:\\n            windows, n, kept = make_windows(tokenizer, row[\'text\'], spec)\\n            entries.extend(windows)\\n            counts.append(len(windows))\\n            original, retained = original + n, retained + kept\\n        vectors, alphas = [], []\\n        cursor = 0\\n        while cursor < len(entries):\\n            try:\\n                values, alpha = encode_batch(model, tokenizer, entries[cursor:cursor + batch_size], spec)\\n            except torch.cuda.OutOfMemoryError:\\n                gc.collect()\\n                torch.cuda.empty_cache()\\n                if batch_size == 1:\\n                    raise\\n                batch_size = max(1, batch_size // 2)\\n                print(f\'CUDA OOM: retry extraction with batch_size={batch_size}\', flush=True)\\n                continue\\n            vectors.extend(values)\\n            alphas.extend(alpha if alpha is not None else [None] * len(values))\\n            cursor += len(values)\\n        documents, explanations, cursor = [], [], 0\\n        for i, (row, count) in enumerate(zip(selected, counts)):\\n            documents.append(np.stack(vectors[cursor:cursor + count]))\\n            if start + i < explain and spec[\'source\'] == \'llama\' and spec[\'pooling\'] == \'cot\':\\n                for j in range(count):\\n                    entry, alpha = entries[cursor + j], alphas[cursor + j]\\n                    positions = [k for k, yes in enumerate(entry[\'content_mask\']) if yes]\\n                    explanations.append(dict(id=row[\'id\'], chunk=j, depth=spec[\'layers\'][0],\\n                        token_start=entry[\'start\'], tokens=tokenizer.convert_ids_to_tokens([entry[\'input_ids\'][k] for k in positions]),\\n                        weights=[float(alpha[k]) for k in positions]))\\n            cursor += count\\n        if torch.cuda.is_available():\\n            torch.cuda.synchronize()\\n        store.append(documents, time.perf_counter() - started, original, retained, explanations)\\n        print(f\'{spec[\\"source\\"]}: cached {store.done}/{len(rows)} documents\', flush=True)\\n    store.finish()\\n\\n\\ndef extract_source(data, source, spec, cache_dir, reuse_cache=None, batch_size=2, read_only=False):\\n    from transformers import AutoConfig, AutoModel, AutoTokenizer, BitsAndBytesConfig, __version__\\n    config = AutoConfig.from_pretrained(spec[\'model\'], revision=spec[\'revision\'], trust_remote_code=False)\\n    spec = dict(spec, source=source, model_commit=getattr(config, \'_commit_hash\', None), transformers=__version__,\\n                feature_format=1, extraction_seed=42,\\n                implementation=digest(Path(__file__).read_text(encoding=\'utf-8\')))\\n    if Path(spec[\'model\']).is_dir():\\n        spec[\'local_model_files\'] = {p.name: [p.stat().st_size, p.stat().st_mtime_ns]\\n                                    for p in Path(spec[\'model\']).iterdir() if p.is_file()}\\n    if source == \'llama\':\\n        spec[\'layers\'] = [config.num_hidden_layers - i * spec[\'stride\'] for i in range(5)]\\n        if min(spec[\'layers\']) < 1:\\n            raise ValueError(\'Backbone is too shallow for five layers at this stride.\')\\n    else:\\n        spec[\'layers\'] = [config.num_hidden_layers]\\n    context = min(getattr(config, \'max_position_embeddings\', 512), 512) if source != \'llama\' else config.max_position_embeddings\\n    # RoBERTa reserves two positional indices for special/padding handling.\\n    if config.model_type == \'roberta\':\\n        context = min(context, config.max_position_embeddings - 2)\\n    spec[\'max_length\'] = min(spec[\'max_length\'], context)\\n    stores = {}\\n    for split, rows in data[\'splits\'].items():\\n        if not rows:\\n            continue\\n        split_spec = dict(spec)\\n        if spec[\'pooling\'] == \'legacy_mean\' and spec[\'task\'] == \'mr\' and split == \'test\':\\n            split_spec[\'max_length\'] = min(1500, context)\\n        signature = dict(spec=split_spec, data=data[\'fingerprint\'], split=split)\\n        key = digest(signature)[:24]\\n        path = Path(cache_dir) / key\\n        if reuse_cache and (Path(reuse_cache) / key / \'state.json\').exists():\\n            saved = json.loads((Path(reuse_cache) / key / \'state.json\').read_text())\\n            if saved.get(\'complete\'):\\n                path = Path(reuse_cache) / key\\n        stores[split] = FeatureStore(path, signature, (len(spec[\'layers\']), config.hidden_size), len(rows))\\n    if all(store.state[\'complete\'] for store in stores.values()):\\n        return stores\\n    if read_only:\\n        raise FileNotFoundError(\'Complete embeddings missing. Run --stage extract (with the same configuration) first.\')\\n    torch.manual_seed(42)\\n    tokenizer = AutoTokenizer.from_pretrained(spec[\'model\'], revision=spec[\'revision\'], trust_remote_code=False,\\n                                              use_fast=config.model_type not in (\'deberta\', \'deberta-v2\'))\\n    if tokenizer.pad_token_id is None:\\n        tokenizer.pad_token = tokenizer.eos_token\\n    if tokenizer.pad_token_id is None:\\n        raise ValueError(\'Tokenizer must define a pad or EOS token.\')\\n    tokenizer.padding_side = \'right\'\\n    gpu = torch.cuda.is_available()\\n    kwargs = dict(config=config, revision=spec[\'revision\'], trust_remote_code=False,\\n                  torch_dtype=torch.float16 if gpu and spec[\'precision\'] != \'fp32\' else torch.float32)\\n    if source == \'llama\':\\n        kwargs[\'attn_implementation\'] = \'sdpa\'\\n        if spec[\'precision\'] == \'nf4\':\\n            if not gpu:\\n                raise ValueError(\'NF4 extraction needs CUDA; use fp32 for CPU smoke tests.\')\\n            kwargs[\'quantization_config\'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=\'nf4\',\\n                bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)\\n        if gpu:\\n            kwargs[\'device_map\'] = \'auto\'\\n            kwargs[\'max_memory\'] = {i: int(torch.cuda.get_device_properties(i).total_memory * 0.80)\\n                                    for i in range(torch.cuda.device_count())}\\n    model = AutoModel.from_pretrained(spec[\'model\'], **kwargs).eval()\\n    if source != \'llama\' and gpu:\\n        model.to(\'cuda:0\')\\n    model.requires_grad_(False)\\n    try:\\n        for split, store in stores.items():\\n            if not store.state[\'complete\']:\\n                extract_rows(model, tokenizer, data[\'splits\'][split], store.state[\'signature\'][\'spec\'], store, batch_size,\\n                             explain=4 if split == \'test\' else 0)\\n    finally:\\n        del model\\n        gc.collect()\\n        if gpu:\\n            torch.cuda.empty_cache()\\n    return stores\\n", "llmembed/models.py": "\\"\\"\\"Table 1 fusion operators and the draft\'s learned chunk/source fusion.\\"\\"\\"\\nimport torch\\nfrom torch import nn\\n\\n\\nclass BaselineFusion(nn.Module):\\n    def __init__(self, classes, dims=(4096, 1024, 1024), strategy=15, hidden=1024, sigma=0.1, author=False):\\n        super().__init__()\\n        if strategy not in range(1, 16):\\n            raise ValueError(\'Table 1 strategy must be 1..15.\')\\n        self.strategy, self.sigma, self.author = strategy, sigma, author\\n        dl, db, dr = dims\\n        sizes = {1: dl, 2: dl, 3: dl, 4: 5 * dl,\\n                 5: dl + db, 6: dl + db, 7: 5 * dl + db,\\n                 8: dl + dr, 9: dl + dr, 10: 5 * dl + dr,\\n                 11: dl + db + dr, 12: dl + db + dr, 13: 5 * dl + db + dr,\\n                 14: 49, 15: 49 + dl}\\n        if strategy >= 14:\\n            self.compress_layers = nn.ModuleList([\\n                nn.Sequential(nn.Linear(dl, db), nn.ReLU(), nn.Dropout(0.5)) for _ in range(5)])\\n            self.roberta_align = nn.Identity() if db == dr else nn.Linear(dr, db)\\n        self.fc1 = nn.Linear(sizes[strategy], hidden)\\n        self.fc2 = nn.Linear(hidden, max(1, hidden // 4))\\n        self.fc3 = nn.Linear(max(1, hidden // 4), classes)\\n        self.dropout = nn.Dropout(0.5)\\n\\n    def fuse(self, llama, bert, roberta):\\n        s = self.strategy\\n        if s >= 14:\\n            sources = torch.stack([self.compress_layers[i](llama[:, i]) for i in range(5)] +\\n                                  [bert, self.roberta_align(roberta)], dim=1)\\n            gram = (sources @ sources.transpose(1, 2)).flatten(1)\\n            # Match released code; the paper\'s tanh(2*sigma*x) equality is a typo.\\n            co = 2 * torch.sigmoid(self.sigma * gram) - 1\\n            return torch.cat([co, llama.mean(1)], 1) if s == 15 else co\\n        if s == 1:\\n            return llama[:, 0]  # Features are ordered deepest to shallowest.\\n        l = llama.flatten(1) if s in (4, 7, 10, 13) else llama.amax(1) if s in (3, 6, 9, 12) else llama.mean(1)\\n        parts = [l]\\n        if s in (5, 6, 7, 11, 12, 13):\\n            parts.append(bert)\\n        if s in (8, 9, 10, 11, 12, 13):\\n            parts.append(roberta)\\n        return torch.cat(parts, 1)\\n\\n    def forward(self, llama, bert, roberta):\\n        x = self.dropout(torch.relu(self.fc1(self.fuse(llama, bert, roberta))))\\n        x = self.dropout(torch.relu(self.fc2(x)))\\n        logits = self.fc3(x)\\n        return (logits.softmax(-1) if self.author else logits), None\\n\\n\\nclass AdaptiveFusion(nn.Module):\\n    def __init__(self, classes, dims=(4096, 1024, 1024), hidden=1024, heads=8, sigma=0.2):\\n        super().__init__()\\n        if hidden % heads or hidden < 4:\\n            raise ValueError(\'Hidden size must be >=4 and divisible by the number of heads.\')\\n        self.sigma = sigma\\n        self.projections = nn.ModuleList([\\n            nn.Sequential(nn.Linear(d, hidden), nn.GELU(), nn.LayerNorm(hidden))\\n            for d in [dims[0]] * 5 + list(dims[1:])])\\n        self.attention = nn.MultiheadAttention(hidden, heads, dropout=0.2, batch_first=True)\\n        self.norm = nn.LayerNorm(hidden)\\n        self.gate = nn.Sequential(nn.Linear(hidden, hidden // 4), nn.Tanh(), nn.Linear(hidden // 4, 1))\\n        self.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(2 * hidden, hidden // 2), nn.GELU(),\\n                                        nn.LayerNorm(hidden // 2), nn.Dropout(0.2), nn.Linear(hidden // 2, classes))\\n\\n    def forward(self, llama, bert, roberta):\\n        inputs = [llama[:, i] for i in range(5)] + [bert, roberta]\\n        x = torch.stack([projection(value) for projection, value in zip(self.projections, inputs)], 1)\\n        attention, _ = self.attention(x, x, x, need_weights=False)\\n        x = self.norm(x + attention)\\n        weights = self.gate(x).softmax(1)\\n        pooled = (weights * x).sum(1)\\n        interaction = torch.tanh(2 * self.sigma * x[:, :5].mean(1) * x[:, 5:].mean(1))\\n        return self.classifier(torch.cat([pooled, interaction], -1)), weights.squeeze(-1)\\n\\n\\nclass ChunkPool(nn.Module):\\n    def __init__(self, dim):\\n        super().__init__()\\n        self.features = nn.Linear(dim, 64)\\n        self.position = nn.Linear(3, 64, bias=False)\\n        self.score = nn.Linear(64, 1, bias=False)\\n\\n    def forward(self, x, mask):\\n        # [batch, chunks, depths, dimension]; tokenizers may yield different chunk counts.\\n        if not mask.any(1).all():\\n            raise ValueError(\'Every document must contain at least one chunk.\')\\n        indices = torch.arange(x.shape[1], device=x.device)[None, :]\\n        last = mask.sum(1, keepdim=True) - 1\\n        positions = torch.stack([(indices / last.clamp_min(1)).expand_as(mask),\\n                                 (indices == 0).expand_as(mask), indices == last], -1).float()\\n        scores = self.score(torch.tanh(self.features(x) + self.position(positions).unsqueeze(2))).squeeze(-1)\\n        weights = scores.masked_fill(~mask.unsqueeze(-1), -torch.inf).softmax(1)\\n        return (weights.unsqueeze(-1) * x).sum(1), weights\\n\\n\\nclass ExperimentModel(nn.Module):\\n    def __init__(self, classes, dims, chunked=False, adaptive=False, strategy=15, hidden=1024, heads=8, sigma=0.1, author=False):\\n        super().__init__()\\n        self.chunked = chunked\\n        self.pools = nn.ModuleList([ChunkPool(dim) for dim in dims]) if chunked else nn.ModuleList()\\n        self.fusion = (AdaptiveFusion(classes, dims, hidden, heads, sigma) if adaptive else\\n                       BaselineFusion(classes, dims, strategy, hidden, sigma, author))\\n\\n    def forward(self, features, masks):\\n        if self.chunked:\\n            pooled = [pool(x, mask) for pool, x, mask in zip(self.pools, features, masks)]\\n            sources, chunks = zip(*pooled)\\n        else:\\n            sources, chunks = [x[:, 0] for x in features], None\\n        logits, gate = self.fusion(sources[0], sources[1][:, 0], sources[2][:, 0])\\n        return logits, gate, chunks\\n", "llmembed/report.py": "\\"\\"\\"Export observed metrics and diagnostic plots; never fill expected results.\\"\\"\\"\\nimport argparse\\nfrom collections import defaultdict\\nimport csv\\nimport json\\nfrom pathlib import Path\\nimport statistics\\n\\nfrom .data import EXPECTED, write_json\\n\\n\\ndef write_csv(path, rows):\\n    if not rows:\\n        Path(path).unlink(missing_ok=True)\\n        return\\n    with Path(path).open(\'w\', encoding=\'utf-8\', newline=\'\') as stream:\\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\\n        writer.writeheader()\\n        writer.writerows(rows)\\n\\n\\ndef make_report(output):\\n    output = Path(output)\\n    records = [json.loads(p.read_text(encoding=\'utf-8\')) for p in sorted((output / \'results\').glob(\'*/result.json\'))]\\n    rows, groups = [], defaultdict(list)\\n    for r in records:\\n        row = {k: r[k] for k in (\'dataset\', \'experiment\', \'seed\', \'benchmark_status\', \'sample_limit\', \'protocol\',\\n            \'precision\', \'best_epoch\', \'parameters\', \'extraction_seconds\', \'training_seconds\', \'inference_seconds\',\\n            \'token_retention\', \'estimated_gpu_kwh\', \'energy_method\', \'data_fingerprint\', \'comparison_key\', \'study_key\', \'run_group\')}\\n        row.update(r[\'metrics\'])\\n        rows.append(row)\\n        groups[r[\'run_group\'], r[\'dataset\'], r[\'experiment\']].append(r)\\n    write_csv(output / \'metrics.csv\', rows)\\n    summaries = []\\n    for (run_group, dataset, name), group in groups.items():\\n        group.sort(key=lambda r: r[\'seed\'])\\n        key = group[0][\'comparison_key\']\\n        values = [r[\'metrics\'][\'accuracy\'] for r in group]\\n        baseline = [r for r in records if r[\'comparison_key\'] == key and r[\'dataset\'] == dataset and r[\'experiment\'] == \'exp0\']\\n        seeds = {r[\'seed\']: r[\'metrics\'][\'accuracy\'] for r in baseline}\\n        paired = [r[\'metrics\'][\'accuracy\'] - seeds[r[\'seed\']] for r in group if r[\'seed\'] in seeds]\\n        summaries.append(dict(dataset=dataset, experiment=name, comparison_key=key, run_group=run_group, study_key=group[0][\'study_key\'],\\n            benchmark_status=group[0][\'benchmark_status\'], sample_limit=group[0][\'sample_limit\'],\\n            precision=group[0][\'precision\'], seeds=\';\'.join(str(r[\'seed\']) for r in group), n_seeds=len(values),\\n            accuracy_mean=statistics.mean(values), accuracy_std=statistics.stdev(values) if len(values) > 1 else None,\\n            macro_f1_mean=statistics.mean(r[\'metrics\'][\'macro_f1\'] for r in group),\\n            delta_exp0_pp=100 * statistics.mean(paired) if len(paired) == len(group) and len(seeds) == len(baseline) else None,\\n            extraction_seconds=group[0][\'extraction_seconds\'],\\n            training_seconds_mean=statistics.mean(r[\'training_seconds\'] for r in group)))\\n    write_csv(output / \'ablation.csv\', summaries)\\n    averages = []\\n    by_study = defaultdict(list)\\n    for row in summaries:\\n        by_study[row[\'study_key\'], row[\'experiment\']].append(row)\\n    for (study, experiment), group in by_study.items():\\n        if len(group) == 5 and {r[\'dataset\'] for r in group} == set(EXPECTED) and len({r[\'seeds\'] for r in group}) == 1:\\n            averages.append(dict(study_key=study, experiment=experiment,\\n                avg5_accuracy=statistics.mean(r[\'accuracy_mean\'] for r in group),\\n                dataset_protocols=\'; \'.join(f\'{r[\\"dataset\\"]}:{r[\\"benchmark_status\\"]}\' for r in group),\\n                sample_limit=group[0][\'sample_limit\'], precision=group[0][\'precision\']))\\n    write_csv(output / \'average_five_datasets.csv\', averages)\\n    import matplotlib\\n    matplotlib.use(\'Agg\')\\n    import matplotlib.pyplot as plt\\n    for r in records:\\n        directory = output / \'results\' / Path(r[\'result_dir\']).name\\n        if r[\'gate_mean\']:\\n            fig, ax = plt.subplots(figsize=(10, 4), layout=\'constrained\')\\n            ax.bar(range(7), r[\'gate_mean\'])\\n            ax.set_xticks(range(7), [name.replace(\'\\\\\\\\\', \'/\').rsplit(\'/\', 1)[-1] for name in r[\'source_names\']], rotation=25, ha=\'right\')\\n            ax.set(ylabel=\'Mean routing weight on test\', title=f\'{r[\\"dataset\\"]} / {r[\\"experiment\\"]} / seed {r[\\"seed\\"]}\')\\n            fig.savefig(directory / \'source_weights.png\', dpi=180)\\n            plt.close(fig)\\n    paths = set((output / \'features\').glob(\'*/state.json\'))\\n    for r in records:\\n        for directory in r[\'feature_dirs\'][\'llama\'].values():\\n            local = output / \'features\' / Path(directory).name / \'state.json\'\\n            path = local if local.exists() else Path(directory) / \'state.json\'\\n            if path.exists():\\n                paths.add(path)\\n    for path in sorted(paths):\\n        state = json.loads(path.read_text(encoding=\'utf-8\'))\\n        examples = {}\\n        for example in state.get(\'explanations\', []):\\n            # shortcut: plot the first chunk per document; all chunk weights remain in state.json.\\n            examples.setdefault(example[\'id\'], example)\\n        for i, example in enumerate(list(examples.values())[:4]):\\n            tokens, weights = example[\'tokens\'], example[\'weights\']\\n            lines = (len(tokens) + 19) // 20\\n            fig, axes = plt.subplots(lines, 1, figsize=(14, max(2, lines * 1.1)), squeeze=False, layout=\'constrained\')\\n            for line, ax in enumerate(axes[:, 0]):\\n                start = line * 20\\n                segment = tokens[start:start + 20]\\n                ax.imshow([weights[start:start + 20]], cmap=\'YlOrRd\', aspect=\'auto\', vmin=0, vmax=max(weights))\\n                ax.set_xticks(range(len(segment)), segment, rotation=30, ha=\'right\', fontsize=8)\\n                ax.set_yticks([])\\n            fig.suptitle(f\'Cosine-guided token weights: {example[\\"id\\"]}, chunk {example[\\"chunk\\"]}, layer {example[\\"depth\\"]}\')\\n            destination = output / \'explanations\' / path.parent.name\\n            destination.mkdir(parents=True, exist_ok=True)\\n            fig.savefig(destination / f\'tokens_{i}.png\', dpi=150)\\n            plt.close(fig)\\n    write_json(output / \'report_notes.json\', dict(\\n        accuracy_std=\'Sample standard deviation across seeds; null with one seed.\',\\n        delta=\'Percentage points vs Exp 0, paired seeds and identical data/training settings.\',\\n        extraction_seconds=\'Original extraction compute for all three sources; excludes model download/load. Reused caches retain their original time.\',\\n        training_seconds=\'Training plus validation, excludes checkpoint serialization.\',\\n        energy=\'No energy measurement. Optional kWh uses user-supplied total GPU watts.\',\\n        explainability=\'Routing/cosine weights are diagnostics, not causal attributions or evidence of generated reasoning.\',\\n        average=\'Produced only with all five datasets and matching seeds. MR proxy remains explicitly identified.\'))\\n    print(f\'Reports: {output / \\"metrics.csv\\"} and {output / \\"ablation.csv\\"}\', flush=True)\\n    return summaries\\n\\n\\nif __name__ == \'__main__\':\\n    p = argparse.ArgumentParser()\\n    p.add_argument(\'output\', type=Path)\\n    make_report(p.parse_args().output)\\n", "llmembed/run.py": "\\"\\"\\"python -m llmembed.run --help\\"\\"\\"\\nimport argparse\\nimport importlib.metadata\\nimport json\\nfrom pathlib import Path\\nimport platform\\nimport time\\n\\nimport torch\\n\\nfrom .data import EXPECTED, prepare_data, write_json\\nfrom .features import extract_source\\nfrom .train import train_run\\n\\n\\ndef parser():\\n    p = argparse.ArgumentParser(description=\'LLMEmbed++ Exp 0-5, with resumable extraction and classifier training.\')\\n    p.add_argument(\'--datasets\', nargs=\'+\', choices=EXPECTED, default=[\'sst2\'])\\n    p.add_argument(\'--experiments\', nargs=\'+\', choices=[\'exp0\', \'exp1\', \'exp2\', \'exp3\', \'exp4\', \'exp5b\', \'exp5c\'],\\n                   default=[\'exp0\', \'exp1\', \'exp2\', \'exp3\', \'exp4\'])\\n    p.add_argument(\'--output\', type=Path, default=Path(\'runs\'))\\n    p.add_argument(\'--data-dir\', type=Path)\\n    p.add_argument(\'--reuse-cache\', type=Path, help=\'Read completed feature directories from a previous Kaggle output.\')\\n    p.add_argument(\'--mr-source\', choices=[\'local\', \'imdb_resplit\'], default=\'local\')\\n    p.add_argument(\'--stage\', choices=[\'data\', \'extract\', \'train\', \'all\'], default=\'all\')\\n    p.add_argument(\'--plan\', action=\'store_true\', help=\'Print experiment matrix without downloads or training.\')\\n    p.add_argument(\'--protocol\', choices=[\'controlled\', \'author\'], default=\'controlled\')\\n    p.add_argument(\'--precision\', choices=[\'nf4\', \'fp16\', \'fp32\'], default=\'nf4\')\\n    p.add_argument(\'--llama-model\', default=\'meta-llama/Llama-2-7b-hf\')\\n    p.add_argument(\'--bert-model\', default=\'google-bert/bert-large-uncased\')\\n    p.add_argument(\'--roberta-model\', default=\'FacebookAI/roberta-large\')\\n    p.add_argument(\'--deberta-model\', default=\'microsoft/deberta-v3-large\')\\n    p.add_argument(\'--next-model\', default=\'Qwen/Qwen2.5-7B\')\\n    p.add_argument(\'--revision\', default=\'main\', help=\'Resolved model commit is recorded in each feature manifest.\')\\n    p.add_argument(\'--limit\', type=int, default=0, help=\'Stratified sample cap per original split; 0 means full dataset.\')\\n    p.add_argument(\'--seeds\', nargs=\'+\', type=int, default=[42, 43, 44])\\n    p.add_argument(\'--split-seed\', type=int, default=42)\\n    p.add_argument(\'--val-fraction\', type=float, default=0.1)\\n    p.add_argument(\'--epochs\', type=int, default=50)\\n    p.add_argument(\'--batch-size\', type=int, default=1024, help=\'Effective classifier batch size.\')\\n    p.add_argument(\'--micro-batch\', type=int, default=64, help=\'Physical batch; gradients accumulate to --batch-size.\')\\n    p.add_argument(\'--extract-batch\', type=int, default=2, help=\'LLM batch; halves automatically after CUDA OOM.\')\\n    p.add_argument(\'--encoder-batch\', type=int, default=16)\\n    p.add_argument(\'--hidden\', type=int, default=1024)\\n    p.add_argument(\'--heads\', type=int, default=8)\\n    p.add_argument(\'--lr\', type=float, default=1e-4)\\n    p.add_argument(\'--weight-decay\', type=float, default=0.01)\\n    p.add_argument(\'--sigma\', type=float, default=0.1)\\n    p.add_argument(\'--gamma\', type=float, default=0.5)\\n    p.add_argument(\'--tau\', type=float, default=0.1)\\n    p.add_argument(\'--stride\', type=int, default=4)\\n    p.add_argument(\'--fusion\', type=int, choices=range(1, 16), default=15)\\n    p.add_argument(\'--fusion-sweep\', action=\'store_true\', help=\'Add all Table 1 fusions using Exp 0 features.\')\\n    p.add_argument(\'--stride-sweep\', action=\'store_true\', help=\'Add Exp 1 with strides 1, 2, 4, 6.\')\\n    p.add_argument(\'--chunk-size\', type=int, default=256, help=\'Content tokens, excluding prompt/special tokens.\')\\n    p.add_argument(\'--overlap\', type=int, default=32)\\n    p.add_argument(\'--max-chunks\', type=int, default=0, help=\'0 retains all chunks; nonzero cap is reported as truncation.\')\\n    p.add_argument(\'--power-watts\', type=float, help=\'Optional assumed TOTAL GPU power for estimated kWh, never measured kWh.\')\\n    return p\\n\\n\\ndef experiment_matrix(args):\\n    experiments = []\\n    for name in args.experiments:\\n        experiments.append(dict(name=name, stride=1 if name == \'exp0\' else args.stride,\\n            cot=name not in (\'exp0\', \'exp1\'), chunked=name in (\'exp3\', \'exp4\', \'exp5b\', \'exp5c\'),\\n            adaptive=name in (\'exp4\', \'exp5b\', \'exp5c\'), fusion=args.fusion,\\n            llama=args.next_model if name == \'exp5c\' else args.llama_model,\\n            bert=args.deberta_model if name in (\'exp5b\', \'exp5c\') else args.bert_model,\\n            roberta=args.roberta_model))\\n    if args.fusion_sweep:\\n        baseline = dict(name=\'exp0\', stride=1, cot=False, chunked=False, adaptive=False,\\n                        fusion=15, llama=args.llama_model, bert=args.bert_model, roberta=args.roberta_model)\\n        experiments.extend(dict(baseline, name=f\'fusion_{i:02}\', fusion=i) for i in range(1, 16))\\n    if args.stride_sweep:\\n        baseline = dict(name=\'exp1\', cot=False, chunked=False, adaptive=False,\\n                        fusion=15, llama=args.llama_model, bert=args.bert_model, roberta=args.roberta_model)\\n        experiments.extend(dict(baseline, name=f\'stride_{i}\', stride=i) for i in (1, 2, 4, 6))\\n    return experiments\\n\\n\\ndef extraction_spec(args, dataset, experiment, source):\\n    llama_lengths = {\'sst2\': 128, \'mr\': 3000, \'agnews\': 256, \'r8\': 1024, \'r52\': 1024}\\n    is_llama = source == \'llama\'\\n    if is_llama:\\n        pooling = \'cot\' if experiment[\'cot\'] else \'legacy_mean\' if args.protocol == \'author\' else \'mean\'\\n    else:\\n        pooling = \'pooler\' if source == \'bert\' and experiment[\'bert\'] == args.bert_model else \'cls\'\\n    return dict(model=experiment[source], revision=args.revision, task=dataset,\\n        stride=experiment[\'stride\'] if is_llama else 1, pooling=pooling,\\n        precision=args.precision if is_llama or args.precision == \'fp32\' else \'fp16\',\\n        max_length=512 if experiment[\'chunked\'] or not is_llama else llama_lengths[dataset],\\n        chunked=experiment[\'chunked\'], chunk_size=args.chunk_size if experiment[\'chunked\'] else 0,\\n        overlap=args.overlap if experiment[\'chunked\'] else 0, max_chunks=args.max_chunks if experiment[\'chunked\'] else 0,\\n        gamma=args.gamma if pooling == \'cot\' else 0.5, tau=args.tau if pooling == \'cot\' else 0.1)\\n\\n\\ndef main(argv=None):\\n    p = parser()\\n    args = p.parse_args(argv)\\n    for name in (\'epochs\', \'batch_size\', \'micro_batch\', \'extract_batch\', \'encoder_batch\', \'hidden\', \'heads\', \'stride\', \'chunk_size\'):\\n        if getattr(args, name) < 1:\\n            p.error(f\'{name} must be positive.\')\\n    if args.limit < 0 or args.max_chunks < 0 or not 0 <= args.overlap < args.chunk_size:\\n        p.error(\'Require limit/max_chunks >= 0 and 0 <= overlap < chunk_size.\')\\n    if args.tau <= 0 or not 0 <= args.gamma <= 1 or args.lr <= 0 or args.sigma <= 0:\\n        p.error(\'Require tau/lr/sigma > 0 and gamma in [0, 1].\')\\n    if args.hidden < 4 or args.hidden % args.heads or args.batch_size % args.micro_batch:\\n        p.error(\'hidden must be >=4 and divisible by heads; batch-size must be divisible by micro-batch.\')\\n    if args.protocol == \'author\':\\n        if args.experiments != [\'exp0\'] or args.fusion_sweep or args.stride_sweep or args.precision != \'fp16\':\\n            p.error(\'Author compatibility requires --experiments exp0 --precision fp16, without sweeps.\')\\n        args.llama_model = \'meta-llama/Llama-2-7b-chat-hf\'\\n        args.val_fraction = 0\\n        if args.hidden != 1024 or args.fusion != 15:\\n            p.error(\'Author compatibility requires hidden=1024 and fusion=15.\')\\n    elif not 0 < args.val_fraction < 1:\\n        p.error(\'Controlled experiments require 0 < val-fraction < 1.\')\\n    if args.power_watts is not None and args.power_watts <= 0:\\n        p.error(\'power-watts must be positive.\')\\n    experiments = experiment_matrix(args)\\n    if args.plan:\\n        print(json.dumps(experiments, indent=2))\\n        return []\\n    if args.stage in (\'all\', \'extract\') and args.precision != \'fp32\' and not torch.cuda.is_available():\\n        p.error(\'CUDA unavailable: select --precision fp32 for CPU, or enable the Kaggle GPU.\')\\n    environment = dict(python=platform.python_version(), torch=torch.__version__,\\n        cuda=torch.version.cuda, gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],\\n        packages={name: importlib.metadata.version(name) for name in (\'numpy\', \'transformers\', \'datasets\', \'accelerate\', \'scikit-learn\')},\\n        args={k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()})\\n    write_json(args.output / \'environment.json\', environment)\\n    results = []\\n    for dataset in args.datasets:\\n        data = prepare_data(dataset, args.data_dir, args.output, limit=args.limit, seed=args.split_seed,\\n                            val_fraction=args.val_fraction, mr_source=args.mr_source)\\n        if args.stage == \'data\':\\n            continue\\n        for experiment in experiments:\\n            print(f\'\\\\n{dataset}: {experiment[\\"name\\"]} ({args.precision}, {args.protocol})\', flush=True)\\n            started = time.perf_counter()\\n            caches = {}\\n            for source in (\'bert\', \'roberta\', \'llama\'):\\n                spec = extraction_spec(args, dataset, experiment, source)\\n                caches[source] = extract_source(data, source, spec, args.output / \'features\', args.reuse_cache,\\n                    args.extract_batch if source == \'llama\' else args.encoder_batch, read_only=args.stage == \'train\')\\n            print(f\'Feature stage wall time (including cache reuse): {time.perf_counter() - started:.1f}s\', flush=True)\\n            if args.stage != \'extract\':\\n                for seed in args.seeds:\\n                    config = {key: getattr(args, key) for key in (\'protocol\', \'epochs\', \'batch_size\', \'micro_batch\',\\n                        \'hidden\', \'heads\', \'lr\', \'weight_decay\', \'sigma\', \'power_watts\')}\\n                    config[\'seed\'] = seed\\n                    config[\'study\'] = {key: getattr(args, key) for key in (\'llama_model\', \'bert_model\', \'roberta_model\',\\n                        \'deberta_model\', \'next_model\', \'precision\', \'revision\', \'stride\', \'gamma\', \'tau\',\\n                        \'chunk_size\', \'overlap\', \'max_chunks\')}\\n                    results.append(train_run(data, caches, experiment, config, args.output))\\n            for source in caches.values():\\n                for store in source.values():\\n                    store.close()\\n    if results:\\n        from .report import make_report\\n        make_report(args.output)\\n    return results\\n\\n\\nif __name__ == \'__main__\':\\n    main()\\n", "llmembed/train.py": "\\"\\"\\"Train only the classifier; select checkpoints using train-derived validation.\\"\\"\\"\\nimport csv\\nimport json\\nfrom pathlib import Path\\nimport random\\nimport time\\n\\nimport numpy as np\\nimport torch\\nfrom torch import nn\\nfrom torch.utils.data import DataLoader, Dataset\\nfrom sklearn.metrics import accuracy_score, f1_score\\n\\nfrom .data import digest, write_json\\nfrom .models import ExperimentModel\\n\\n\\nclass CachedDataset(Dataset):\\n    def __init__(self, stores, rows):\\n        self.stores, self.rows = stores, rows\\n        if any(store.done != len(rows) for store in stores):\\n            raise ValueError(\'Feature/label document counts differ.\')\\n\\n    def __len__(self):\\n        return len(self.rows)\\n\\n    def __getitem__(self, index):\\n        return [store[index] for store in self.stores], self.rows[index][\'label\']\\n\\n\\ndef collate(batch):\\n    features, labels = zip(*batch)\\n    tensors, masks = [], []\\n    for source in range(3):\\n        arrays = [item[source] for item in features]\\n        longest = max(len(a) for a in arrays)\\n        tensor = torch.zeros(len(batch), longest, *arrays[0].shape[1:], dtype=torch.float32)\\n        mask = torch.zeros(len(batch), longest, dtype=torch.bool)\\n        for i, array in enumerate(arrays):\\n            tensor[i, :len(array)] = torch.from_numpy(np.array(array, dtype=np.float32, copy=True))\\n            mask[i, :len(array)] = True\\n        tensors.append(tensor)\\n        masks.append(mask)\\n    return tensors, masks, torch.tensor(labels, dtype=torch.long)\\n\\n\\ndef seed_everything(seed):\\n    random.seed(seed)\\n    np.random.seed(seed)\\n    torch.manual_seed(seed)\\n    if torch.cuda.is_available():\\n        torch.cuda.manual_seed_all(seed)\\n    torch.backends.cudnn.benchmark = False\\n    torch.backends.cudnn.deterministic = True\\n\\n\\ndef to_device(batch, device):\\n    features, masks, labels = batch\\n    return [x.to(device) for x in features], [x.to(device) for x in masks], labels.to(device)\\n\\n\\n@torch.inference_mode()\\ndef evaluate(model, loader, device, classes):\\n    model.eval()\\n    predictions, labels, gates, chunk_examples = [], [], [], []\\n    total_loss = 0.0\\n    for batch in loader:\\n        features, masks, target = to_device(batch, device)\\n        logits, gate, chunks = model(features, masks)\\n        total_loss += nn.functional.cross_entropy(logits, target, reduction=\'sum\').item()\\n        predictions.extend(logits.argmax(-1).cpu().tolist())\\n        labels.extend(target.cpu().tolist())\\n        if gate is not None:\\n            gates.append(gate.cpu())\\n        if chunks is not None and len(chunk_examples) < 4:\\n            for i in range(min(len(target), 4 - len(chunk_examples))):\\n                chunk_examples.append([weights[i, masks[s][i]].cpu().tolist() for s, weights in enumerate(chunks)])\\n    metrics = dict(loss=total_loss / len(labels), accuracy=accuracy_score(labels, predictions),\\n                   macro_f1=f1_score(labels, predictions, labels=list(range(classes)), average=\'macro\', zero_division=0),\\n                   micro_f1=f1_score(labels, predictions, labels=list(range(classes)), average=\'micro\', zero_division=0))\\n    return metrics, predictions, torch.cat(gates).mean(0).tolist() if gates else None, chunk_examples\\n\\n\\ndef save_checkpoint(path, content):\\n    temporary = path.with_suffix(\'.tmp\')\\n    torch.save(content, temporary)\\n    temporary.replace(path)\\n\\n\\ndef train_run(data, caches, experiment, config, output_dir):\\n    implementation = digest([Path(__file__).read_text(encoding=\'utf-8\'),\\n                             Path(__file__).with_name(\'models.py\').read_text(encoding=\'utf-8\')])\\n    signature = dict(data=data[\'fingerprint\'], experiment=experiment, training=config,\\n                     features=[caches[s][\'train\'].state[\'signature\'] for s in (\'llama\', \'bert\', \'roberta\')],\\n                     training_format=1, implementation=implementation)\\n    target = Path(output_dir) / \'results\' / f\'{data[\\"dataset\\"]}-{experiment[\\"name\\"]}-{digest(signature)[:16]}\'\\n    target.mkdir(parents=True, exist_ok=True)\\n    result_path = target / \'result.json\'\\n    if result_path.exists():\\n        return json.loads(result_path.read_text(encoding=\'utf-8\'))\\n    seed_everything(config[\'seed\'])\\n    classes = len(data[\'label_names\'])\\n    dims = [caches[s][\'train\'].shape[-1] for s in (\'llama\', \'bert\', \'roberta\')]\\n    device = torch.device(\'cuda:0\' if torch.cuda.is_available() else \'cpu\')\\n    author = config[\'protocol\'] == \'author\'\\n    model = ExperimentModel(classes, dims, chunked=experiment[\'chunked\'], adaptive=experiment[\'adaptive\'],\\n        strategy=experiment[\'fusion\'], hidden=config[\'hidden\'], heads=config[\'heads\'], sigma=config[\'sigma\'], author=author).to(device)\\n    optimizer = (torch.optim.Adam(model.parameters(), lr=config[\'lr\']) if author else\\n                 torch.optim.AdamW(model.parameters(), lr=config[\'lr\'], weight_decay=config[\'weight_decay\']))\\n    scheduler = None if author else torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, config[\'epochs\'])\\n    datasets = {split: CachedDataset([caches[s][split] for s in (\'llama\', \'bert\', \'roberta\')], rows)\\n                for split, rows in data[\'splits\'].items() if rows}\\n    if not author and \'val\' not in datasets:\\n        raise ValueError(\'Controlled protocol requires a train-derived validation split.\')\\n    batch_size = config[\'micro_batch\']\\n    effective = config[\'batch_size\']\\n    if batch_size < 1 or effective % batch_size:\\n        raise ValueError(\'Effective batch size must be divisible by micro batch size.\')\\n    val_loader = DataLoader(datasets[\'val\'], batch_size=batch_size, collate_fn=collate) if \'val\' in datasets else None\\n    history, first_epoch, best, best_epoch, elapsed = [], 0, -1.0, 0, 0.0\\n    last_path, best_path = target / \'last.pt\', target / \'best.pt\'\\n    if last_path.exists():\\n        state = torch.load(last_path, map_location=\'cpu\', weights_only=True)\\n        model.load_state_dict(state[\'model\'])\\n        optimizer.load_state_dict(state[\'optimizer\'])\\n        if scheduler is not None:\\n            scheduler.load_state_dict(state[\'scheduler\'])\\n        torch.set_rng_state(state[\'rng\'])\\n        if device.type == \'cuda\' and state[\'cuda_rng\']:\\n            torch.cuda.set_rng_state_all(state[\'cuda_rng\'])\\n        first_epoch, history, best, best_epoch, elapsed = state[\'epoch\'], state[\'history\'], state[\'best\'], state[\'best_epoch\'], state[\'seconds\']\\n    write_json(target / \'config.json\', signature)\\n    for epoch in range(first_epoch, config[\'epochs\']):\\n        if device.type == \'cuda\':\\n            torch.cuda.synchronize()\\n        started = time.perf_counter()\\n        generator = torch.Generator().manual_seed(config[\'seed\'] + epoch)\\n        loader = DataLoader(datasets[\'train\'], batch_size=batch_size, shuffle=True, generator=generator, collate_fn=collate)\\n        model.train()\\n        seen, loss_sum, correct = 0, 0.0, 0\\n        optimizer.zero_grad(set_to_none=True)\\n        for batch in loader:\\n            features, masks, labels = to_device(batch, device)\\n            if seen % effective == 0:\\n                group_size = min(effective, len(datasets[\'train\']) - seen)\\n            logits, _, _ = model(features, masks)\\n            loss = nn.functional.cross_entropy(logits, labels, reduction=\'sum\')\\n            if not torch.isfinite(loss):\\n                raise FloatingPointError(\'Non-finite training loss; cached data/checkpoint preserved.\')\\n            (loss / group_size).backward()\\n            seen += len(labels)\\n            loss_sum += loss.item()\\n            correct += (logits.argmax(-1) == labels).sum().item()\\n            if seen % effective == 0 or seen == len(datasets[\'train\']):\\n                optimizer.step()\\n                optimizer.zero_grad(set_to_none=True)\\n        validation = evaluate(model, val_loader, device, classes)[0] if val_loader else None\\n        if scheduler is not None:\\n            scheduler.step()\\n        if device.type == \'cuda\':\\n            torch.cuda.synchronize()\\n        elapsed += time.perf_counter() - started\\n        history.append(dict(epoch=epoch + 1, loss=loss_sum / seen, accuracy=correct / seen, validation=validation))\\n        score = validation[\'accuracy\'] if validation else float(epoch)\\n        if score > best:\\n            best, best_epoch = score, epoch + 1\\n            save_checkpoint(best_path, model.state_dict())\\n        save_checkpoint(last_path, dict(model=model.state_dict(), optimizer=optimizer.state_dict(),\\n            scheduler=scheduler.state_dict() if scheduler else None, rng=torch.get_rng_state(),\\n            cuda_rng=torch.cuda.get_rng_state_all() if device.type == \'cuda\' else [], epoch=epoch + 1,\\n            history=history, best=best, best_epoch=best_epoch, seconds=elapsed))\\n        print(f\'{data[\\"dataset\\"]}/{experiment[\\"name\\"]} seed={config[\\"seed\\"]} epoch={epoch + 1}: \'\\n              f\'train={correct / seen:.4f} val={validation[\\"accuracy\\"] if validation else \\"unused\\"}\', flush=True)\\n    model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True))\\n    test_loader = DataLoader(datasets[\'test\'], batch_size=batch_size, collate_fn=collate)\\n    started = time.perf_counter()\\n    metrics, predictions, gate_mean, chunks = evaluate(model, test_loader, device, classes)\\n    if device.type == \'cuda\':\\n        torch.cuda.synchronize()\\n    inference_seconds = time.perf_counter() - started\\n    stores = [store for source in caches.values() for store in source.values()]\\n    original = sum(s.state[\'original_tokens\'] for s in stores)\\n    retained = sum(s.state[\'retained_tokens\'] for s in stores)\\n    result = dict(dataset=data[\'dataset\'], experiment=experiment[\'name\'], seed=config[\'seed\'],\\n        benchmark_status=data[\'benchmark_status\'], sample_limit=data[\'limit\'], protocol=config[\'protocol\'],\\n        precision=caches[\'llama\'][\'train\'].state[\'signature\'][\'spec\'][\'precision\'],\\n        data_fingerprint=data[\'fingerprint\'], comparison_key=digest([data[\'fingerprint\'], dict(config, seed=None), implementation]),\\n        run_group=digest(dict(signature, training=dict(config, seed=None))),\\n        study_key=digest([dict(config, seed=None), data[\'limit\'], data[\'val_fraction\'], data[\'split_seed\'], implementation]),\\n        metrics=metrics, best_epoch=best_epoch, training_seconds=elapsed, inference_seconds=inference_seconds,\\n        extraction_seconds=sum(s.state[\'seconds\'] for s in stores), token_retention=retained / original,\\n        retention_by_source={s: sum(v.state[\'retained_tokens\'] for v in splits.values()) /\\n                               sum(v.state[\'original_tokens\'] for v in splits.values()) for s, splits in caches.items()},\\n        parameters=sum(p.numel() for p in model.parameters()), gate_mean=gate_mean,\\n        feature_dirs={source: {split: str(store.path) for split, store in splits.items()} for source, splits in caches.items()},\\n        source_names=[f\'LLM layer {d}\' for d in caches[\'llama\'][\'train\'].state[\'signature\'][\'spec\'][\'layers\']] +\\n                     [caches[\'bert\'][\'train\'].state[\'signature\'][\'spec\'][\'model\'], caches[\'roberta\'][\'train\'].state[\'signature\'][\'spec\'][\'model\']],\\n        chunk_examples=chunks, result_dir=str(target), history=history)\\n    if config.get(\'power_watts\') is not None:\\n        result[\'estimated_gpu_kwh\'] = config[\'power_watts\'] * (result[\'extraction_seconds\'] + elapsed + inference_seconds) / 3_600_000\\n        result[\'energy_method\'] = f\'assumed total GPU power={config[\\"power_watts\\"]} W; not measured\'\\n    else:\\n        result.update(estimated_gpu_kwh=None, energy_method=\'not measured\')\\n    with (target / \'predictions.csv\').open(\'w\', encoding=\'utf-8\', newline=\'\') as stream:\\n        writer = csv.writer(stream)\\n        writer.writerow([\'id\', \'label\', \'prediction\'])\\n        writer.writerows((r[\'id\'], r[\'label\'], p) for r, p in zip(data[\'splits\'][\'test\'], predictions))\\n    write_json(result_path, result)\\n    del model, optimizer\\n    if device.type == \'cuda\':\\n        torch.cuda.empty_cache()\\n    return result\\n", "tests/test_experiments.py": "import json\\nfrom pathlib import Path\\nimport sys\\nimport tempfile\\nimport unittest\\nfrom unittest.mock import patch\\n\\nimport numpy as np\\n\\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\\nfrom llmembed.data import split_indices, normalize_records, prepare_data\\nimport torch\\nfrom llmembed.models import BaselineFusion, AdaptiveFusion, ChunkPool, ExperimentModel\\nfrom llmembed.features import token_pool, window_ranges, make_windows, FeatureStore\\nfrom llmembed.run import main, parser, experiment_matrix, extraction_spec\\n\\ntorch.set_num_threads(1)\\n\\n\\nclass DataTests(unittest.TestCase):\\n    def test_split_preserves_rare_classes_and_never_overlaps(self):\\n        labels = [0] * 20 + [1] * 3 + [2]\\n        train, val = split_indices(labels, 0.2, 42)\\n        self.assertFalse(set(train) & set(val))\\n        self.assertEqual(sorted(train + val), list(range(len(labels))))\\n        self.assertEqual(set(np.array(labels)[train]), {0, 1, 2})\\n        self.assertEqual((train, val), split_indices(labels, 0.2, 42))\\n\\n    def test_bad_local_data_fails_before_extraction(self):\\n        with self.assertRaises(ValueError):\\n            normalize_records({\'text\': [\'hello\'], \'label\': [0, 1]})\\n        with self.assertRaises(ValueError):\\n            normalize_records([{\'text\': \' \', \'label\': 0}])\\n        with self.assertRaises(ValueError):\\n            normalize_records([{\'text\': \'hello\', \'label\': -1}])\\n        with self.assertRaises(ValueError):\\n            normalize_records({\'text\': \'abc\', \'label\': [0, 1, 0]})\\n\\n    def test_local_data_shared_label_map_and_fingerprint(self):\\n        with tempfile.TemporaryDirectory() as directory:\\n            root = Path(directory)\\n            source = root / \'input\' / \'r8\'\\n            source.mkdir(parents=True)\\n            rows = [{\'text\': f\'doc {i}\', \'label\': \'b\' if i % 2 else \'a\'} for i in range(20)]\\n            (source / \'train.json\').write_text(json.dumps(rows))\\n            (source / \'test.json\').write_text(json.dumps(rows[:2]))\\n            first = prepare_data(\'r8\', root / \'input\', root / \'out\', limit=0)\\n            second = prepare_data(\'r8\', root / \'input\', root / \'out\', limit=0)\\n            self.assertEqual(first[\'fingerprint\'], second[\'fingerprint\'])\\n            self.assertEqual(first[\'label_names\'], [\'a\', \'b\'])\\n            import shutil\\n            shutil.copytree(root / \'input\', root / \'moved\')\\n            moved = prepare_data(\'r8\', root / \'moved\', root / \'out\', limit=0)\\n            self.assertEqual(first[\'fingerprint\'], moved[\'fingerprint\'])\\n            self.assertEqual(len(first[\'splits\'][\'test\']), 2)\\n            self.assertFalse(set(r[\'id\'] for r in first[\'splits\'][\'train\']) &\\n                             set(r[\'id\'] for r in first[\'splits\'][\'val\']))\\n            rows[0][\'text\'] = \'changed document\'\\n            (source / \'train.json\').write_text(json.dumps(rows))\\n            third = prepare_data(\'r8\', root / \'input\', root / \'out\', limit=0)\\n            self.assertNotEqual(first[\'fingerprint\'], third[\'fingerprint\'])\\n\\n\\nclass ModelTests(unittest.TestCase):\\n    def test_all_fifteen_fusions_and_adaptive_backward(self):\\n        inputs = (torch.randn(3, 5, 12), torch.randn(3, 8), torch.randn(3, 8))\\n        for strategy in range(1, 16):\\n            model = BaselineFusion(4, (12, 8, 8), strategy=strategy, hidden=8)\\n            logits, _ = model(*inputs)\\n            self.assertEqual(logits.shape, (3, 4))\\n            torch.nn.functional.cross_entropy(logits, torch.tensor([0, 1, 2])).backward()\\n        model = AdaptiveFusion(4, (12, 8, 6), hidden=8, heads=2)\\n        logits, gate = model(inputs[0], inputs[1], torch.randn(3, 6))\\n        torch.testing.assert_close(gate.sum(1), torch.ones(3))\\n        logits.sum().backward()\\n        self.assertIsNotNone(model.projections[0][0].weight.grad)\\n\\n    def test_cooccurrence_matches_author_normalization(self):\\n        model = BaselineFusion(2, (12, 8, 8), strategy=15, hidden=8).eval()\\n        l, b, r = torch.randn(2, 5, 12), torch.randn(2, 8), torch.randn(2, 8)\\n        sources = torch.stack([model.compress_layers[i](l[:, i]) for i in range(5)] + [b, r], 1)\\n        expected = torch.cat([2 * torch.sigmoid(0.1 * (sources @ sources.transpose(1, 2)).flatten(1)) - 1,\\n                              l.mean(1)], 1)\\n        torch.testing.assert_close(model.fuse(l, b, r), expected)\\n\\n    def test_chunk_padding_has_no_effect_and_learns(self):\\n        pool = ChunkPool(8)\\n        x = torch.randn(2, 4, 1, 8)\\n        mask = torch.tensor([[True, True, False, False], [True, False, False, False]])\\n        result, weights = pool(x, mask)\\n        modified = x.clone()\\n        modified[~mask] = 100000\\n        other, _ = pool(modified, mask)\\n        torch.testing.assert_close(result, other)\\n        torch.testing.assert_close(weights.sum(1), torch.ones(2, 1))\\n        self.assertEqual(weights[~mask].sum().item(), 0)\\n        result.square().sum().backward()\\n        self.assertIsNotNone(pool.score.weight.grad)\\n\\n\\nclass FeatureTests(unittest.TestCase):\\n    def test_truncation_preserves_prompt_anchor_and_content_only_mask(self):\\n        from transformers import BertTokenizerFast\\n        with tempfile.TemporaryDirectory() as directory:\\n            path = Path(directory) / \'vocab.txt\'\\n            path.write_text(\'\\\\n\'.join([\'[PAD]\', \'[UNK]\', \'[CLS]\', \'[SEP]\', \'[MASK]\', \'film\', \':\', \'\\"\', \'.\']))\\n            tokenizer = BertTokenizerFast(vocab_file=str(path))\\n            spec = dict(pooling=\'cot\', task=\'sst2\', max_length=48, chunked=False,\\n                        chunk_size=16, overlap=2, max_chunks=0)\\n            entries, original, kept = make_windows(tokenizer, \'film \' * 200, spec)\\n            entry = entries[0]\\n            self.assertEqual(len(entry[\'input_ids\']), 48)\\n            self.assertLess(kept, original)\\n            self.assertEqual(sum(entry[\'content_mask\']), kept)\\n            self.assertEqual(entry[\'input_ids\'][entry[\'anchor\']], tokenizer.convert_tokens_to_ids(\':\'))\\n            self.assertFalse(entry[\'content_mask\'][entry[\'anchor\']])\\n            entries, original, kept = make_windows(tokenizer, \'film \' * 200, dict(spec, chunked=True))\\n            self.assertEqual(original, kept)\\n            self.assertTrue(all(len(e[\'input_ids\']) <= 48 for e in entries))\\n\\n    def test_windows_cover_every_token_including_tail(self):\\n        for n in (1, 224, 256, 257, 448, 2470):\\n            intervals = window_ranges(n, 256, 32)\\n            covered = {t for a, b in intervals for t in range(a, b)}\\n            self.assertEqual(covered, set(range(n)))\\n            self.assertTrue(all(0 < b - a <= 256 for a, b in intervals))\\n        with self.assertRaises(ValueError):\\n            window_ranges(50, 32, 32)\\n\\n    def test_pool_masks_prompt_padding_and_anchor(self):\\n        h = torch.tensor([[[100., 100.], [1., 0.], [0., 1.], [1., 1.], [999., 999.]]])\\n        attention = torch.tensor([[1, 1, 1, 1, 0]])\\n        content = torch.tensor([[0, 1, 1, 0, 0]], dtype=torch.bool)\\n        pooled, weights = token_pool(h, attention, content, torch.tensor([3]), \'cot\', 0.5, 0.1)\\n        torch.testing.assert_close(weights, torch.tensor([[0., .5, .5, 0., 0.]]))\\n        torch.testing.assert_close(pooled, torch.tensor([[.75, .75]]))\\n        mean, _ = token_pool(h, attention, content, torch.tensor([3]), \'mean\', 0.5, 0.1)\\n        torch.testing.assert_close(mean, h[:, :4].mean(1))\\n        with self.assertRaises(ValueError):\\n            token_pool(h, attention, content, torch.tensor([3]), \'cot\', .5, 0)\\n\\n    def test_feature_store_recovers_uncommitted_tail(self):\\n        with tempfile.TemporaryDirectory() as directory:\\n            store = FeatureStore(Path(directory), {\'key\': \'same\'}, (5, 4), 2)\\n            store.append([np.ones((2, 5, 4), dtype=np.float16)], 1.0, 30, 30, [])\\n            with (Path(directory) / \'values.bin\').open(\'ab\') as f:\\n                f.write(b\'interrupted write\')\\n            resumed = FeatureStore(Path(directory), {\'key\': \'same\'}, (5, 4), 2)\\n            resumed.append([np.zeros((1, 5, 4), dtype=np.float16)], 2.0, 10, 8, [])\\n            resumed.finish()\\n            self.assertEqual(resumed[0].shape, (2, 5, 4))\\n            self.assertEqual(resumed[1].sum(), 0)\\n            self.assertEqual(resumed.state[\'original_tokens\'], 40)\\n            self.assertEqual(resumed.state[\'retained_tokens\'], 38)\\n            with self.assertRaises(ValueError):\\n                FeatureStore(Path(directory), {\'key\': \'different\'}, (5, 4), 2)\\n            resumed.close()\\n\\n\\nclass PipelineTests(unittest.TestCase):\\n    def test_experiments_do_not_confound_exp2_and_exp3(self):\\n        args = parser().parse_args([])\\n        exps = {e[\'name\']: e for e in experiment_matrix(args)}\\n        self.assertTrue(exps[\'exp2\'][\'cot\'])\\n        self.assertFalse(exps[\'exp2\'][\'chunked\'])\\n        self.assertTrue(exps[\'exp3\'][\'chunked\'])\\n        self.assertFalse(exps[\'exp3\'][\'adaptive\'])\\n        self.assertTrue(exps[\'exp4\'][\'adaptive\'])\\n        self.assertEqual(extraction_spec(args, \'sst2\', exps[\'exp3\'], \'llama\'),\\n                         extraction_spec(args, \'sst2\', exps[\'exp4\'], \'llama\'))\\n        self.assertEqual(extraction_spec(args, \'sst2\', exps[\'exp0\'], \'bert\'),\\n                         extraction_spec(args, \'sst2\', exps[\'exp2\'], \'bert\'))\\n\\n    def test_offline_all_experiments_resume_and_exports(self):\\n        from transformers import BertConfig, BertModel, BertTokenizerFast, LlamaConfig, LlamaModel\\n        from llmembed import train\\n        with tempfile.TemporaryDirectory() as directory:\\n            root = Path(directory)\\n            vocab = [\'[PAD]\', \'[UNK]\', \'[CLS]\', \'[SEP]\', \'[MASK]\', \'good\', \'bad\', \'film\', \'news\', \'text\', \':\', \'\\"\', \'.\']\\n            vocab_path = root / \'vocab.txt\'\\n            vocab_path.write_text(\'\\\\n\'.join(vocab), encoding=\'utf-8\')\\n            tokenizer = BertTokenizerFast(vocab_file=str(vocab_path))\\n            models = {}\\n            for name, dim, depth in [(\'llama\', 16, 32), (\'bert\', 8, 2), (\'roberta\', 8, 2),\\n                                     (\'deberta\', 12, 2), (\'next\', 12, 28)]:\\n                path = root / name\\n                if name in (\'llama\', \'next\'):\\n                    model = LlamaModel(LlamaConfig(vocab_size=len(vocab), hidden_size=dim, intermediate_size=dim * 2,\\n                        num_hidden_layers=depth, num_attention_heads=2, num_key_value_heads=2, max_position_embeddings=128))\\n                else:\\n                    model = BertModel(BertConfig(vocab_size=len(vocab), hidden_size=dim, intermediate_size=dim * 2,\\n                        num_hidden_layers=depth, num_attention_heads=2, max_position_embeddings=128))\\n                model.save_pretrained(path)\\n                tokenizer.save_pretrained(path)\\n                models[name] = path\\n            source = root / \'data\' / \'sst2\'\\n            source.mkdir(parents=True)\\n            for split, count in [(\'train\', 16), (\'test\', 4)]:\\n                rows = [dict(text=((\'good\' if i % 2 else \'bad\') + \' film \') * (i + 2), label=i % 2) for i in range(count)]\\n                (source / f\'{split}.json\').write_text(json.dumps(rows))\\n            args = [\'--data-dir\', str(root / \'data\'), \'--output\', str(root / \'run\'), \'--datasets\', \'sst2\',\\n                \'--experiments\', \'exp0\', \'exp1\', \'exp2\', \'exp3\', \'exp4\', \'exp5b\', \'exp5c\', \'--precision\', \'fp32\',\\n                \'--seeds\', \'7\', \'--epochs\', \'2\', \'--hidden\', \'16\', \'--heads\', \'2\', \'--micro-batch\', \'2\',\\n                \'--batch-size\', \'4\', \'--chunk-size\', \'8\', \'--overlap\', \'2\']\\n            for name, path in models.items():\\n                args.extend([f\'--{name}-model\', str(path)])\\n            original_save = train.save_checkpoint\\n\\n            def interrupt_after_first_epoch(path, content):\\n                original_save(path, content)\\n                if path.name == \'last.pt\' and content[\'epoch\'] == 1:\\n                    raise RuntimeError(\'simulated session interruption\')\\n\\n            with patch.object(train, \'save_checkpoint\', side_effect=interrupt_after_first_epoch):\\n                with self.assertRaisesRegex(RuntimeError, \'simulated session\'):\\n                    main(args)\\n            results = main(args)\\n            self.assertEqual(len(results), 7)\\n            self.assertEqual(len(results[0][\'history\']), 2)\\n            for result in results:\\n                self.assertTrue(0 <= result[\'metrics\'][\'accuracy\'] <= 1)\\n                self.assertEqual(result[\'estimated_gpu_kwh\'], None)\\n            self.assertEqual(results[4][\'retention_by_source\'], dict(llama=1., bert=1., roberta=1.))\\n            self.assertTrue((root / \'run\' / \'metrics.csv\').exists())\\n            self.assertTrue((Path(results[4][\'result_dir\']) / \'source_weights.png\').exists())\\n            self.assertTrue(list((root / \'run\' / \'explanations\').glob(\'*/tokens_0.png\')))\\n            self.assertFalse((root / \'run\' / \'average_five_datasets.csv\').exists())\\n            states = list((root / \'run\' / \'features\').glob(\'*/state.json\'))\\n            before = {str(p): p.stat().st_mtime_ns for p in states}\\n            again = main(args + [\'--stage\', \'train\'])\\n            self.assertEqual(before, {str(p): p.stat().st_mtime_ns for p in states})\\n            self.assertEqual(results[0][\'metrics\'], again[0][\'metrics\'])\\n            clean = main(args + [\'--output\', str(root / \'clean\'), \'--experiments\', \'exp0\',\\n                                 \'--reuse-cache\', str(root / \'run\' / \'features\')])\\n            resumed = torch.load(Path(results[0][\'result_dir\']) / \'last.pt\', map_location=\'cpu\', weights_only=True)\\n            uninterrupted = torch.load(Path(clean[0][\'result_dir\']) / \'last.pt\', map_location=\'cpu\', weights_only=True)\\n            for key, tensor in resumed[\'model\'].items():\\n                torch.testing.assert_close(tensor, uninterrupted[\'model\'][key])\\n\\n\\nif __name__ == \'__main__\':\\n    unittest.main()\\n", "requirements.txt": "# Keep Kaggle\'s CUDA-enabled torch installation. CPU tests require torch>=2.6.\\ntransformers==4.51.3\\ndatasets==3.5.0\\naccelerate==1.6.0\\nbitsandbytes==0.50.2; sys_platform == \'linux\'\\nsentencepiece>=0.2,<0.3\\nprotobuf>=3.20\\nnumpy>=1.26,<3\\nscikit-learn>=1.3,<2\\nmatplotlib>=3.7,<4\\n", "README.md": "# LLMEmbed++: thực nghiệm trên Kaggle\\n\\nTriển khai bản thảo `BanThao_LLMEmbed.md`, đối chiếu Bảng 1–3 và mục Limitations của bài LLMEmbed ACL 2024. Mã gốc trong `LLMEmbed-ACL2024/`, notebook SST-2 cũ và log cũ được giữ nguyên. Notebook mới: **`llmembed_plusplus.ipynb`**.\\n\\n## Bắt đầu trên Kaggle\\n\\n1. Import `llmembed_plusplus.ipynb` vào một Kaggle Notebook, bật **Internet** và **GPU T4 ×2**.\\n2. Xin quyền dùng [Llama-2-7b-hf](https://huggingface.co/meta-llama/Llama-2-7b-hf), tạo token Hugging Face có quyền đọc, thêm vào Kaggle Secrets tên **`HF_TOKEN`** và cấp quyền cho notebook. Không dán token vào cell.\\n3. Chạy mặc định `RUN_MODE=\'smoke\'`: SST-2, tối đa 256 mẫu mỗi split, 2 epochs, seed 42. Đây chỉ là kiểm tra vận hành.\\n4. Khi chạy được, đổi `RUN_MODE=\'full\'`: toàn bộ dữ liệu, 50 epochs, seeds 42/43/44. Chọn **một dataset mỗi phiên**, và chọn các experiment phù hợp thời gian/dung lượng còn lại.\\n5. `PRECISION=\'nf4\'` tiết kiệm VRAM; `PRECISION=\'fp16\'` dùng model không lượng tử hóa. Giữ cùng precision cho toàn bộ chuỗi ablation; NF4 không phải tái lập số liệu FP16 của bài báo.\\n6. Save Version để lưu output trước khi phiên kết thúc. Các tensor 7B không nằm trong notebook/ZIP; được tải khi chạy.\\n\\nNotebook tự giải nén mã nguồn đã nhúng, không cần upload repo hay dữ liệu trước. Các lệnh `subprocess.run(..., check=True)` dừng khi có lỗi. Chưa có token LLaMA thì có thể chạy riêng `EXPERIMENTS=[\'exp5c\']` với Qwen công khai; kết quả này thuộc Exp 5C.\\n\\n## Các thực nghiệm\\n\\n| ID | 5 tầng LLM (thứ tự sâu → nông) | Trích xuất | Gộp đoạn | Fusion |\\n|---|---|---|---|---|\\n| exp0 | 32,31,30,29,28 | Raw, masked mean | Một đoạn bị giới hạn context | Table 1 #15 |\\n| exp1 | 32,28,24,20,16 | Raw, masked mean | Như exp0 | #15 |\\n| exp2 | 32,28,24,20,16 | Prompt + anchor/cosine | Một đoạn; giữ nguyên suffix | #15 |\\n| exp3 | 32,28,24,20,16 | Như exp2 | Sliding windows + learned attention | #15 |\\n| exp4 / Exp 5A | 32,28,24,20,16 | Như exp3 | Như exp3 | AG-CMF |\\n| exp5b | LLaMA-2 như exp4 | BERT → DeBERTa-v3-large | Như exp3 | AG-CMF |\\n| exp5c | Qwen2.5-7B: 28,24,20,16,12 | Qwen + DeBERTa + RoBERTa | Như exp3 | AG-CMF |\\n\\nChiều embedding và số tầng lấy từ config model, không cố định 4096/32 cho Qwen. Có thể chọn `--next-model meta-llama/Meta-Llama-3-8B` khi có quyền truy cập; bộ chọn tầng vẫn lấy 5 tầng từ cuối với stride chỉ định.\\n\\n- BERT dùng `pooler_output`, RoBERTa dùng token đầu như code gốc. DeBERTa dùng token đầu, vì không có BERT pooler.\\n- `gamma=0.5`, `tau=0.1`, chunk 256 **content tokens**, overlap 32. Mỗi tokenizer chia đoạn riêng; mỗi nguồn gộp đoạn riêng rồi mới fusion.\\n- Không giới hạn số đoạn mặc định (`--max-chunks 0`). Nếu chủ động đặt cap, lấy các cửa sổ trải đều và báo tỷ lệ token thực sự được giữ.\\n- Learned chunk attention thêm vị trí tương đối, cờ đoạn đầu/cuối. AG-CMF gồm 7 phép chiếu riêng, MHA, cổng softmax và tương tác Hadamard; mặc định K=1024, 8 heads.\\n- Exp 3 và 4 dùng chung cache. BERT/RoBERTa không chia đoạn được tái sử dụng giữa Exp 0/1/2. Các nguồn/chế độ khác có cache riêng.\\n\\n## Dữ liệu và so sánh công bằng\\n\\n| Dataset | Nguồn tự tải | Train/test gốc | Ý nghĩa |\\n|---|---|---|---|\\n| SST-2 | `stanfordnlp/sst2` | 67.349 / 872 | Dùng split validation có nhãn làm test, theo bài LLMEmbed |\\n| AGNews | `fancyzhx/ag_news` | 120.000 / 7.600 | Split train/test công khai |\\n| R8 | TextGCN, commit cố định | 5.485 / 2.189 | Đúng số mẫu; chưa xác nhận tiền xử lý trùng tác giả |\\n| R52 | TextGCN, commit cố định | 6.532 / 2.568 | Đúng số mẫu; chưa xác nhận tiền xử lý trùng tác giả |\\n| MR proxy | `stanfordnlp/imdb`, chia lại seed 42 | 40.000 / 10.000 | **Không phải bộ chia MR đã được tác giả xác nhận** |\\n\\nRepo không chứa dữ liệu MR và không mô tả cách tạo split 40k/10k. IMDb gốc là 25k/25k. `--mr-source imdb_resplit` ghép hai split **có nhãn**, chia stratified 80/20, không dùng 50k mẫu unsupervised. Cờ này phải được chọn tường minh khi dùng CLI; notebook đã hiển thị rõ lựa chọn proxy. Không thay bằng `rotten_tomatoes` hay MR 10.662 câu và gọi là cùng benchmark.\\n\\nCó dữ liệu tác giả thì dùng `--data-dir /kaggle/input/my-data --mr-source local`, cấu trúc:\\n\\n```text\\nmy-data/\\n  mr/train.json\\n  mr/test.json\\n  r8/train.json\\n  r8/test.json\\n  r52/train.json\\n  r52/test.json\\n```\\n\\nChấp nhận `[{\'text\': \'...\', \'label\': 0}, ...]` hoặc `{\'text\': [...], \'label\': [...]}` trong JSON chuẩn (dấu ngoặc kép), hoặc JSONL mỗi dòng một record. Nhãn chuỗi được ánh xạ chung theo tập train. Từ chối nhãn test lạ, nhãn -1, văn bản trống, và độ dài mảng không khớp. Dữ liệu local luôn được ưu tiên nếu thư mục dataset tồn tại. `benchmark_status=local_split_unverified` là ghi nhận nguồn, không tự chứng nhận nó khớp paper.\\n\\n**Protocol mặc định `controlled`:** tách stratified 10% từ train làm validation bằng seed 42, giữ các lớp singleton trong train. Tất cả experiment/training seed dùng cùng split. Huấn luyện AdamW, LR 1e-4, weight decay 0.01, cosine scheduler, effective batch 1024. Gradient accumulation với micro batch 64 giảm VRAM; dropout khiến nó không đồng nhất từng bit với một forward batch 1024. Chọn checkpoint có validation accuracy tốt nhất (giữ epoch đầu khi hòa), đánh giá test một lần sau huấn luyện. Accuracy và macro/micro F1 tính trên toàn bộ mẫu, không lấy trung bình metric từng batch. Không dùng SST-2 validation/test để chọn epoch.\\n\\n**Protocol `author`:** chạy riêng Exp 0, chat checkpoint như mã gốc trong repo, mean gồm padding cố định, softmax trước CrossEntropy như mã gốc, Adam, toàn bộ train, epoch cuối, không validation/scheduler. Đây là chế độ tương thích đường tính và mục tiêu huấn luyện của code tác giả; precision, môi trường và dữ liệu vẫn có thể khác, không đảm bảo khớp từng bit hoặc đạt đúng 0.9620. Code sửa cho `controlled` xuất logits trực tiếp.\\n\\n```bash\\npython -m llmembed.run --datasets sst2 --experiments exp0 --protocol author --precision fp16 --epochs 100 --seeds 42 --output runs/author\\n```\\n\\n`controlled` giữ lại validation nên số mẫu thực sự dùng để cập nhật head ít hơn bảng paper; so sánh đóng góp bằng **baseline tự chạy cùng protocol**. Báo cáo số liệu paper riêng, không chép 0.9620 vào hàng kết quả thực nghiệm.\\n\\n## Lệnh CLI\\n\\nChạy từ thư mục `kaggle_experiment` sau khi cài requirements. Dùng torch có CUDA trên Kaggle; kiểm thử tại máy chỉ cần torch >=2.6 CPU.\\n\\n```bash\\n# Xem kế hoạch, không tải model/dữ liệu\\npython -m llmembed.run --plan\\n\\n# Chạy thử\\npython -m llmembed.run --datasets sst2 --limit 256 --epochs 2 --seeds 42 --output runs/smoke\\n\\n# Ablation đầy đủ một dataset\\npython -m llmembed.run --datasets r52 --experiments exp0 exp1 exp2 exp3 exp4 --precision fp16 --output runs/full\\n\\n# MR proxy, được ghi nhãn riêng trong mọi kết quả\\npython -m llmembed.run --datasets mr --mr-source imdb_resplit --output runs/full\\n\\n# Khảo sát backbone\\npython -m llmembed.run --datasets sst2 --experiments exp4 exp5b exp5c --output runs/full\\n\\n# 15 phép fusion trong Bảng 1 và 4 stride; tái sử dụng cache\\npython -m llmembed.run --datasets r52 --experiments exp0 --fusion-sweep --stride-sweep --output runs/sweeps\\n\\n# Tách extraction và training; các tham số còn lại phải giống nhau\\npython -m llmembed.run --datasets r52 --stage extract --output runs/full\\npython -m llmembed.run --datasets r52 --stage train --output runs/full\\n\\n# Tổng hợp lại các result.json đã có\\npython -m llmembed.report runs/full\\n```\\n\\n`--sigma`, `--gamma`, `--tau`, `--stride`, `--chunk-size`, `--hidden` được lưu trong cấu hình. Muốn chọn hyperparameter thì dùng **validation**, cố định lựa chọn trước khi công bố test; việc chạy sweep không tự chọn “best test”.\\n\\n## Lưu trữ, chạy tiếp, bộ nhớ\\n\\n- Nạp lần lượt từng backbone, tắt gradient và KV cache, dùng `AutoModel` để không tạo logits từ vựng. Dynamic padding trong controlled. LLM FP16 có thể được Accelerate chia trên hai GPU; NF4 có thể vừa một GPU.\\n- Batch extraction bắt đầu 2 cho LLM/16 cho encoder, giảm một nửa khi CUDA OOM, dừng với lỗi nếu batch 1 vẫn không đủ. Model-load OOM không tự thay backbone hoặc precision. Giảm batch, dùng NF4 hoặc chọn GPU phù hợp.\\n- Feature float16 lưu phẳng `values.bin` và offsets, đọc bằng memory map; RAM không chứa toàn bộ feature. Sau mỗi 32 documents ghi state atomically. Khởi động lại sẽ bỏ phần đuôi chưa commit và tiếp tục từ document đã lưu.\\n- Cache key gồm SHA256 dữ liệu/split, model/revision/commit, tokenizer theo model đó, precision, pooling/layers/prompt/chunk, version extractor. Không trộn cache smoke/full hay các cấu hình khác nhau.\\n- Checkpoint `last.pt` lưu optimizer, scheduler, RNG và epoch; `best.pt` dùng đánh giá. Chạy lại cùng lệnh tiếp tục từ epoch hoàn tất. Thay số epochs tạo run mới vì cosine schedule thay đổi.\\n- `--reuse-cache /kaggle/input/old-output/features` đọc cache hoàn tất tại input chỉ đọc. Để tiếp tục cache dở hoặc training, sao chép output cũ sang thư mục working bằng `RESUME_DIR` trong notebook rồi chạy cùng cấu hình. Không chạy đồng thời hai tiến trình vào cùng output.\\n- Một cache LLaMA không chia đoạn tốn khoảng **40 KiB/document** (5×4096×2 bytes): AGNews train+test khoảng **4,87 GiB**; chunking nhân thêm số đoạn. Nhiều experiment + model weights + checkpoints có thể vượt dung lượng phiên. Chạy theo dataset/experiment, Save Version rồi gắn output làm input khi cần.\\n\\n## Kết quả xuất ra\\n\\n- `environment.json`: phiên bản Python/thư viện/CUDA/GPU và lệnh chạy; không có HF token.\\n- `data/*/manifest.json`: nguồn, SHA256, ánh xạ nhãn, counts, split seed, cảnh báo văn bản trùng train/test.\\n- `features/*/state.json`: model commit, progress, thời gian extraction, tỷ lệ token gốc giữ lại theo tokenizer, token weights.\\n- `results/*/`: config, best/last checkpoint, history, `predictions.csv`, metric test, mean gate, trọng số đoạn của 4 mẫu đầu test.\\n- `metrics.csv`: từng seed; `ablation.csv`: mean/sample std và delta theo **điểm phần trăm**, chỉ ghép baseline cùng split/cấu hình/seed. Một seed thì std để trống.\\n- `average_five_datasets.csv`: chỉ tạo khi đủ 5 datasets và cùng seeds. Cột protocol vẫn nêu MR proxy; số trung bình này không tự trở thành đối chiếu chính xác với paper.\\n- `results/*/source_weights.png`, `explanations/*/tokens_*.png`: trọng số cổng trên test và heatmap cosine token. Vẽ đoạn đầu của tối đa 4 document; toàn bộ trọng số các đoạn của 4 document đó vẫn lưu trong state JSON. Trọng số token không phải giải thích nhân quả hay chuỗi suy luận được sinh.\\n- Thời gian extraction là compute ban đầu của các cache, không tính tải/nạp model. Training time gồm train + validation, không gồm ghi checkpoint; inference đo riêng. Khi dùng lại cache, không diễn giải thời gian trích xuất đã lưu thành thời gian phiên hiện tại.\\n- Điện năng mặc định **không đo**. `--power-watts 140` chỉ tạo **ước lượng** với giả định tổng công suất 140 W; không ghi số đó là phép đo, không bao gồm CPU/toàn hệ thống.\\n\\n## Các điểm cần sửa khi diễn giải bản thảo\\n\\n1. Co-occurrence của code tác giả là `2*sigmoid(sigma*X)-1 = tanh(sigma*X/2)`. Biểu thức `tanh(2*sigma*X)` trong paper/bản thảo không tương đương. Baseline ở đây dùng đúng code tác giả; nhánh Hadamard AG-CMF giữ công thức riêng trong bản thảo.\\n2. Cosine token pooling với gamma/tau cố định **không được học**. Chỉ chunk attention, projections, MHA, gate và classifier được học. Single-pass prompt không chứng minh có CoT reasoning.\\n3. Mask chỉ cho token văn bản gốc tham gia cosine pool, loại prompt/special/padding/anchor; suffix luôn được giữ. Văn bản được token hóa riêng rồi ghép token IDs với prefix/suffix để kiểm soát budget, không decode/re-tokenize các đoạn.\\n4. Tám đoạn 256, overlap 32 chỉ phủ tối đa 1.824 content tokens. Muốn tránh cắt, dùng số đoạn không giới hạn; tỷ lệ giữ token không đồng nghĩa giữ mọi quan hệ ngữ cảnh dài giữa các đoạn.\\n5. R52=0.4217 trong Table 2 thuộc RoBERTa và một số cấu hình Cat; cấu hình Cat ba nguồn đạt **0.9326**. Bản thảo đã đảo MR/AGNews ở vài hàng (RoBERTa, IO, Cat, Cat+Co). Cần chép lại từ PDF khi lập bảng đối thủ.\\n6. Không có bảo đảm trước về accuracy >0.968/0.973, tăng tốc 8–10× hoặc <0.25 kWh. Các giá trị đó là mục tiêu, không phải kết quả đã được kiểm chứng.\\n7. Table 1 #1–14 được triển khai theo mô tả fusion và dùng cùng MLP 1024→256; repo chỉ cung cấp head #15 nên không khẳng định kiến trúc head của 14 cấu hình còn lại được tác giả công bố đầy đủ.\\n\\n## Kiểm tra và tái tạo notebook\\n\\n```bash\\npython -m unittest discover -s tests -v\\npython make_plusplus_notebook.py\\n```\\n\\nTests dùng mô hình tiny ngẫu nhiên lưu local: đủ Exp 0–5, kích thước backbone khác nhau, forward/backward 15 fusion, che padding/prompt, chunk coverage, recovery cache, checkpoint resume, cache reuse và CSV/PNG. Không tải model 7B và không xác nhận chất lượng phân loại. Build notebook nhúng chính các file Python đã kiểm tra; notebook gốc không bị ghi đè.\\n\\n## Nguồn đối chiếu\\n\\n- [LLMEmbed, ACL 2024](https://aclanthology.org/2024.acl-long.433/) và [mã tác giả](https://github.com/ChunLiu-cs/LLMEmbed-ACL2024).\\n- [TextGCN: corpus và split Reuters](https://github.com/yao8839836/text_gcn/tree/962223652e9bb164ac2d83cd09fc7b8845ce860b/data).\\n- [SST-2](https://huggingface.co/datasets/stanfordnlp/sst2), [AGNews](https://huggingface.co/datasets/fancyzhx/ag_news), [IMDb](https://huggingface.co/datasets/stanfordnlp/imdb).\\n- [Transformers 4.51.3: bitsandbytes](https://huggingface.co/docs/transformers/v4.51.3/en/quantization/bitsandbytes), [DeBERTa](https://huggingface.co/docs/transformers/v4.51.3/en/model_doc/deberta-v2), [Qwen2.5-7B config](https://huggingface.co/Qwen/Qwen2.5-7B/blob/main/config.json).\\n"}')
for name, content in FILES.items():
    path = CODE / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
os.chdir(CODE)
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
print('Code:', CODE)


In [ ]:
# Cài thư viện trước khi import transformers. Giữ torch/CUDA có sẵn của Kaggle.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements.txt')], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Bật GPU trong Settings của notebook rồi chạy lại.')
# The previous Kaggle run loaded CPU-only bitsandbytes despite CUDA torch.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall',
                '--no-deps', '--index-url', 'https://pypi.org/simple', 'bitsandbytes==0.50.2'], check=True)
subprocess.run([sys.executable, '-c',
    'import importlib.metadata; from bitsandbytes.cextension import lib; '
    'assert lib.compiled_with_cuda, "bitsandbytes CUDA backend unavailable"; '
    'print("bitsandbytes:", importlib.metadata.version("bitsandbytes"))'], check=True)
print('torch:', torch.__version__)
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


In [ ]:
# Token chỉ ở biến môi trường, không ghi vào notebook, config hay kết quả.
if not os.environ.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
        print('Đã nạp HF_TOKEN từ Kaggle Secrets.')
    except Exception:
        print('Chưa có HF_TOKEN. LLaMA-2 cần quyền truy cập; Exp 5c dùng Qwen công khai.')


## Cấu hình chạy

- `smoke`: kiểm tra pipeline; số liệu không dùng báo cáo nghiên cứu.
- `full`: toàn bộ train/test, 50 epochs, 3 seeds. Nên chạy **một dataset mỗi phiên**.
- Exp 5A chính là Exp 4; chọn thêm `exp5b`, `exp5c` để đổi backbone.
- `mr` tải IMDb và chia lại 40k/10k khi `MR_SOURCE='imdb_resplit'`. **Đây là proxy**, chưa phải bộ chia MR của tác giả.
- R8/R52 tải bản TextGCN có đúng số mẫu; chưa xác nhận tiền xử lý trùng repo LLMEmbed.


In [ ]:
RUN_MODE = 'smoke'  # 'full' khi đã kiểm tra chạy được
DATASETS = ['sst2']  # 'mr', 'agnews', 'r8', 'r52'
EXPERIMENTS = ['exp0', 'exp1', 'exp2', 'exp3', 'exp4']
PRECISION = 'nf4'  # 'fp16' trên T4 x2 để so sánh không lượng tử hóa
PROTOCOL = 'controlled'
MR_SOURCE = 'imdb_resplit'
DATA_DIR = None  # Ví dụ '/kaggle/input/my-data'; chứa mr/train.json, mr/test.json
REUSE_CACHE = None  # Ví dụ '/kaggle/input/previous-run/llmembed_runs/full/features'
RESUME_DIR = None  # Thư mục output phiên cũ nếu cần tiếp tục cả classifier
STAGE = 'all'  # 'data', 'extract', 'train', 'all'
OUTPUT = Path('/kaggle/working/llmembed_runs') / RUN_MODE
LIMIT = 256 if RUN_MODE == 'smoke' else 0
EPOCHS = 2 if RUN_MODE == 'smoke' else 50
SEEDS = [42] if RUN_MODE == 'smoke' else [42, 43, 44]

command = [sys.executable, '-u', '-m', 'llmembed.run',
    '--datasets', *DATASETS, '--experiments', *EXPERIMENTS,
    '--output', str(OUTPUT), '--precision', PRECISION, '--protocol', PROTOCOL,
    '--mr-source', MR_SOURCE, '--limit', str(LIMIT), '--epochs', str(EPOCHS),
    '--seeds', *map(str, SEEDS), '--stage', STAGE,
    '--extract-batch', '2', '--encoder-batch', '16', '--micro-batch', '64',
    '--chunk-size', '256', '--overlap', '32', '--max-chunks', '0']
if DATA_DIR:
    command += ['--data-dir', DATA_DIR]
if REUSE_CACHE:
    command += ['--reuse-cache', REUSE_CACHE]
if not os.environ.get('HF_TOKEN') and any(experiment != 'exp5c' for experiment in EXPERIMENTS):
    raise RuntimeError('Bật HF_TOKEN trong Add-ons → Secrets trước khi chạy LLaMA-2.')
print(' '.join(command))
subprocess.run(command + ['--plan'], check=True)


In [ ]:
# Tùy chọn: tiếp tục output đã lưu ở một Kaggle Dataset.
if RESUME_DIR:
    import shutil
    if OUTPUT.exists() and any(OUTPUT.iterdir()):
        raise RuntimeError('Chọn OUTPUT trống trước khi khôi phục phiên cũ để tránh ghi đè kết quả.')
    shutil.copytree(RESUME_DIR, OUTPUT, dirs_exist_ok=True)

# Kiểm tra bằng mô hình tí hon, không tải model 7B.
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(CODE / 'tests'), '-v'], check=True)


In [ ]:
# Run: dừng ngay khi có lỗi; cache/checkpoint đã hoàn thành vẫn được giữ.
subprocess.run(command, check=True)


In [ ]:
# Kết quả thật; accuracy ở thang 0..1.
import pandas as pd
from IPython.display import display, Image
if (OUTPUT / 'ablation.csv').exists():
    display(pd.read_csv(OUTPUT / 'ablation.csv'))
    display(pd.read_csv(OUTPUT / 'metrics.csv'))
    for path in list((OUTPUT / 'results').glob('*/source_weights.png'))[:3]:
        display(Image(filename=str(path)))
    for path in list((OUTPUT / 'explanations').glob('*/tokens_0.png'))[:2]:
        display(Image(filename=str(path)))
print('Lưu toàn bộ output bằng Save Version để dùng lại cache/checkpoint:', OUTPUT)


## Đọc kết quả và chạy tiếp

`metrics.csv`: từng seed; `ablation.csv`: mean/std và delta theo điểm phần trăm so với Exp 0.
`average_five_datasets.csv` chỉ có khi chạy đủ 5 dataset cùng cấu hình/seeds.
`results/*/best.pt`, `last.pt`, `predictions.csv`, `config.json` lưu mô hình, dự đoán và cấu hình.
`features/*/state.json` lưu nguồn model, tỷ lệ token giữ lại và tiến độ cache.

Để tiếp tục sau khi Kaggle kết thúc phiên: Save Version, thêm output cũ bằng Add Input,
đặt `RESUME_DIR` hoặc `REUSE_CACHE` ở trên. Cache chưa hoàn thành cần được sao chép vào OUTPUT ghi được.
`STAGE='train'` chỉ dùng cache hoàn thành, không trích xuất lại.

Trọng số token là cosine cố định, không phải token attention được học; prompt không sinh chuỗi suy luận.
Tốc độ/độ chính xác trong bản thảo là giả thuyết cần đo. Điện năng mặc định để trống.
Xem `README.md` trong thư mục code để chạy sweep 15 fusion, 4 strides và chế độ tương thích mã tác giả.
